# arc3-v22-ab-tp10 — single-variable A/B: stock vs TP10 on the 27B V22 serving stack (one boot, two 25-game phases)

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["keithtyser/taaf-duck-qwen38-serving-v1", "driessmit1/arc3-vllm-h100-wheelhouse-v3"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")


# V22 only modifies the vLLM launch flags inside the bundled setup command.
# The target model owns native MTP tensors and the original checkpoint validation
# already verifies that mtp.* tensors are mounted.
def _v31_serving_commands(command: str) -> tuple[str, str, bool]:
    if "vllm.entrypoints.openai.api_server" not in command:
        return command, command, False

    source_block = """        '--kv-cache-dtype',
        'fp8',
    ]"""

    v22_block = """        '--kv-cache-dtype',
        'fp8',
        '--speculative-config',
        '{"method":"mtp","num_speculative_tokens":3}',
        '--async-scheduling',
    ]"""

    v31_block = """        '--kv-cache-dtype',
        'fp8',
        '--speculative-config',
        '{"method":"mtp","num_speculative_tokens":3}',
        '--async-scheduling',
        '--no-enable-chunked-prefill',
    ]"""

    if source_block not in command:
        print("V31: known source serving block not found; leaving it unchanged.", flush=True)
        return command, command, False

    return (
        command.replace(source_block, v31_block, 1),
        command.replace(source_block, v22_block, 1),
        True,
    )


def _v22_cleanup_partial_server() -> None:
    import signal

    pid_path = WORKING_DIR / "vllm-openai-server.pid"
    if not pid_path.exists():
        return
    try:
        pid = int(pid_path.read_text(encoding="utf-8").strip())
        try:
            os.kill(pid, signal.SIGTERM)
            time.sleep(2)
        except OSError:
            pass
        try:
            os.kill(pid, 0)
        except OSError:
            pass
        else:
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
    except Exception as exc:
        print(f"V22: partial-server cleanup warning: {exc!r}", flush=True)
    finally:
        pid_path.unlink(missing_ok=True)


# Solver setup commands run before the benchmark loads.
# Primary = exact V22 MTP3 stack + no-chunked-prefill.
# Fallback = exact MTP3+async serving that produced 2.66 LB.
env = _command_env()
for original_command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    primary_command, v22_command, optimized = _v31_serving_commands(original_command)

    if not optimized:
        print(f"taaf.kaggle: setup command: {original_command}", flush=True)
        subprocess.run(original_command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    else:
        print("V31: launching MTP3 + async + FP8 KV + no-chunked-prefill.", flush=True)
        try:
            subprocess.run(primary_command, shell=True, check=True, cwd=WORKING_DIR, env=env)
        except subprocess.CalledProcessError as exc:
            print(
                f"V31: primary startup failed ({exc!r}); "
                "falling back to exact V22 MTP3+async.",
                flush=True,
            )
            _v22_cleanup_partial_server()
            subprocess.run(v22_command, shell=True, check=True, cwd=WORKING_DIR, env=env)

    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# ---- V31 runtime resilience ----
import threading as _v31_threading
import urllib.request as _v31_urllib

_V31_URL = os.environ.get("LOCAL_ANALYZER_BASE_URL", "http://127.0.0.1:1234/v1").rstrip("/")
_V31_PID = WORKING_DIR / "vllm-openai-server.pid"
_V31_LOG = WORKING_DIR / "vllm-openai-server.log"
_V31_STOP = _v31_threading.Event()
_V31_THREAD = None
_V31_LOCK = _v31_threading.Lock()


def _v31_healthy(timeout: float = 4.0) -> bool:
    try:
        with _v31_urllib.urlopen(f"{_V31_URL}/models", timeout=timeout) as response:
            return 200 <= int(response.status) < 500
    except Exception:
        return False


def _v31_wait_healthy(timeout: float = 480.0) -> bool:
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if _v31_healthy(5.0):
            return True
        time.sleep(5)
    return False


def _v31_kill() -> None:
    if not _V31_PID.exists():
        return
    try:
        pid = int(_V31_PID.read_text(encoding="utf-8").strip())
        try:
            os.kill(pid, signal.SIGTERM)
            time.sleep(2)
        except OSError:
            pass
        try:
            os.kill(pid, 0)
        except OSError:
            pass
        else:
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
    except Exception as exc:
        print(f"V31 cleanup warning: {exc!r}", flush=True)
    finally:
        _V31_PID.unlink(missing_ok=True)


def _v31_spawn(no_chunked: bool) -> None:
    provenance_file = WORKING_DIR / "qwen38-model-provenance.json"
    provenance = json.loads(provenance_file.read_text(encoding="utf-8"))
    model_path = str(provenance["model_path"])

    site_packages = WORKING_DIR / "vllm-site-packages"
    child_env = os.environ.copy()
    current_pp = child_env.get("PYTHONPATH", "")
    if str(site_packages) not in current_pp.split(os.pathsep):
        child_env["PYTHONPATH"] = (
            str(site_packages)
            if not current_pp
            else str(site_packages) + os.pathsep + current_pp
        )
    child_env.update({
        "USE_TF": "0",
        "TRANSFORMERS_NO_TF": "1",
        "TRANSFORMERS_NO_TORCHVISION": "1",
        "VLLM_NO_USAGE_STATS": "1",
    })

    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", model_path,
        "--served-model-name", os.environ.get("INFERENCE_ANALYZER_MODEL", "Qwen/Qwen3.8-27B-FP8"),
        "--host", "127.0.0.1",
        "--port", "1234",
        "--tensor-parallel-size", "1",
        "--enable-auto-tool-choice",
        "--tool-call-parser", "qwen3_coder",
        "--generation-config", "vllm",
        "--enable-prefix-caching",
        "--default-chat-template-kwargs", '{"preserve_thinking": true}',
        "--reasoning-parser", "qwen3",
        "--max-model-len", "262144",
        "--kv-cache-dtype", "fp8",
        "--speculative-config", '{"method":"mtp","num_speculative_tokens":3}',
        "--async-scheduling",
    ]
    if no_chunked:
        cmd.append("--no-enable-chunked-prefill")

    # Prevent unbounded log growth across restarts.
    if _V31_LOG.exists():
        previous = WORKING_DIR / "vllm-openai-server.previous.log"
        try:
            previous.unlink(missing_ok=True)
            _V31_LOG.replace(previous)
        except OSError:
            pass

    log_handle = _V31_LOG.open("w", encoding="utf-8")
    process = subprocess.Popen(
        cmd,
        env=child_env,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,
    )
    _V31_PID.write_text(str(process.pid), encoding="utf-8")

    if not _v31_wait_healthy():
        raise RuntimeError("restarted vLLM did not become healthy in time")


def _v31_recover() -> bool:
    with _V31_LOCK:
        if _v31_healthy():
            return True

        print("V31 watchdog: recovering MTP3/no-chunk server.", flush=True)
        _v31_kill()
        try:
            _v31_spawn(no_chunked=True)
            print("V31 watchdog: primary recovery succeeded.", flush=True)
            return True
        except Exception as first:
            print(f"V31 watchdog: primary recovery failed: {first!r}", flush=True)

        _v31_kill()
        try:
            _v31_spawn(no_chunked=False)
            print("V31 watchdog: exact V22 MTP3 recovery succeeded.", flush=True)
            return True
        except Exception as second:
            print(f"V31 watchdog: fallback recovery failed: {second!r}", flush=True)
            _v31_kill()
            return False


def _v31_start_watchdog(solver) -> None:
    global _V31_THREAD
    _V31_STOP.clear()

    def worker():
        failures = 0
        while not _V31_STOP.wait(15):
            if _v31_healthy():
                failures = 0
                continue

            failures += 1
            print(f"V31 watchdog: health failure {failures}/3", flush=True)
            if failures < 3:
                continue

            if _v31_recover():
                failures = 0
                continue

            print(
                "V31 watchdog: server unrecoverable; stopping solver to preserve partial score.",
                flush=True,
            )
            stop_event = getattr(solver, "_stop_event", None)
            if stop_event is not None:
                stop_event.set()
            return

    _V31_THREAD = _v31_threading.Thread(
        target=worker,
        name="v31-watchdog",
        daemon=True,
    )
    _V31_THREAD.start()


def _v31_stop_watchdog() -> None:
    _V31_STOP.set()
    if _V31_THREAD is not None:
        _V31_THREAD.join(timeout=5)


if not _v31_healthy(10):
    raise RuntimeError("V31 preflight: local vLLM API is not healthy.")
print("V31 preflight: local vLLM API healthy.", flush=True)



In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [ ]:
# V31 keeps the V22 gameplay/prompt/tool policy unchanged.
bm.solver.save_request_logs = False
print("V31 concurrency:", getattr(bm.solver, "concurrency", None))
print("V31 analyzer_timeout:", getattr(bm.solver, "analyzer_timeout", None))
print("V31 save_request_logs:", getattr(bm.solver, "save_request_logs", None))


In [ ]:
# ==== graft install (A/B machinery; every flag phase-controlled) ====
# All grafts install ONCE; behaviour is env-gated per phase. Defaults ALL OFF —
# the stock phase must be byte-equivalent stock behaviour.
import importlib as _il

for _flag in ['TP_ENABLE', 'TP2_ENABLE', 'TP4_ENABLE', 'TP5_ENABLE', 'TP6_ENABLE', 'TP7_ENABLE', 'TP8_ENABLE', 'TP9_ENABLE', 'TP10_ENABLE']:
    os.environ[_flag] = "0"

_G_DIR = WORKING_DIR / "graft_bundle"
_G_DIR.mkdir(parents=True, exist_ok=True)
_GRAFT_SOURCES = {'graft_throughput.py': '"""Throughput graft (Pack 1, 2026-08-29) — plumbing-only changes that raise\nactions-per-game of the anim-bundle ToolAgent. No prompt text changes.\n\nEvidence: docs/research-2026-08-29/R4-harness-throughput-audit.md — every\nplay dies on the 7,920 s clock at ~88 actions/game; each call re-prefills a\n16-20k-token prompt (prefix-cache hit 0-20%) because the trimmer drops one\nblock per turn; the 60 s yield is shorter than one call (26% of wall in\naction-less slices); carried notes are wiped on every GAME_OVER; blind\n20-140-action batches burn efficiency and trigger GAME_OVERs.\n\nSeams (anim bundle inference/agent/tool_agent.py, the code that plays at eval):\n- _trim_messages_for_context (:2056) — hysteresis: when over budget, cut to\n  TP_TRIM_LOW_WATER x budget in one go so the cached prefix survives turns.\n- __init__ — _context_budget_tokens / _yield_seconds / _tool_steps are set\n  from module constants read at import; override per instance from\n  TP_CONTEXT_WINDOW / TP_YIELD_SECONDS / TP_TOOL_STEPS.\n- _update_summarized_knowledge_from_step_summary (:1343) — wipes the carried\n  notes on game_over; keep them (TP_KEEP_NOTES_ON_GAME_OVER). Level-up and\n  run-complete wipes are untouched.\n- _run_python_tool (:1693) + _normalize_python_actions (:1614) — per tool\n  call action budget (TP_BATCH_CAP): requests beyond the cap are truncated,\n  and a further action() call raises ValueError inside the sandbox so the\n  model sees a readable error instead of a blind 100-action walk.\n\nFlags (read at call time; TP_ENABLE=0 turns every seam into a pass-through):\n  TP_ENABLE=1  TP_TRIM_LOW_WATER=0.5  TP_CONTEXT_WINDOW=24576\n  TP_YIELD_SECONDS=900  TP_TOOL_STEPS=8  TP_KEEP_NOTES_ON_GAME_OVER=1\n  TP_BATCH_CAP=10\n\nFail-open: graft logic errors fall through to the stock method; the stock\nmethod is never wrapped in try/except, so its own errors propagate as stock.\n"""\nfrom __future__ import annotations\n\nimport os\nimport threading\nfrom typing import Any\n\n_tls = threading.local()\n_STATE = {"installed": False}\n# Pack 2 hook: called as ON_CUT(agent, dropped_messages) after a hysteresis cut.\nON_CUT = None\n\nDEFAULT_TRIM_LOW_WATER = 0.5\nDEFAULT_CONTEXT_WINDOW = 24576\nDEFAULT_YIELD_SECONDS = 900.0\nDEFAULT_TOOL_STEPS = 8\nDEFAULT_BATCH_CAP = 10\nBATCH_CAP_MESSAGE = (\n    "action batch cap reached: at most {cap} actions per python tool call. "\n    "Observe the results so far, then call the tool again for more actions."\n)\n_OFF = {"0", "false", "no", "off"}\n\n\n# ----------------------------------------------------------------- flags ---\ndef _env(name: str, default: str) -> str:\n    raw = os.environ.get(name)\n    return default if raw is None or not raw.strip() else raw.strip()\n\n\ndef enabled() -> bool:\n    return _env("TP_ENABLE", "1").lower() not in _OFF\n\n\ndef trim_low_water() -> float:\n    """Fraction of the budget to cut down to when over budget; 1.0 = stock."""\n    if not enabled():\n        return 1.0\n    try:\n        value = float(_env("TP_TRIM_LOW_WATER", str(DEFAULT_TRIM_LOW_WATER)))\n    except ValueError:\n        return DEFAULT_TRIM_LOW_WATER\n    return min(1.0, max(0.05, value))\n\n\ndef context_window() -> int:\n    """Context window for the request budget; 0 = leave stock."""\n    if not enabled():\n        return 0\n    try:\n        return max(0, int(_env("TP_CONTEXT_WINDOW", str(DEFAULT_CONTEXT_WINDOW))))\n    except ValueError:\n        return DEFAULT_CONTEXT_WINDOW\n\n\ndef yield_seconds() -> float:\n    """Turn yield in seconds; -1 = leave stock; 0 = disable the yield."""\n    if not enabled():\n        return -1.0\n    try:\n        return float(_env("TP_YIELD_SECONDS", str(DEFAULT_YIELD_SECONDS)))\n    except ValueError:\n        return DEFAULT_YIELD_SECONDS\n\n\ndef tool_steps() -> int:\n    """Calls per turn; -1 = leave stock; 0 = unlimited."""\n    if not enabled():\n        return -1\n    try:\n        return int(_env("TP_TOOL_STEPS", str(DEFAULT_TOOL_STEPS)))\n    except ValueError:\n        return DEFAULT_TOOL_STEPS\n\n\ndef tool_timeout() -> int:\n    """Python sandbox timeout override; 0 = leave stock (clamped 30 s)."""\n    if not enabled():\n        return 0\n    try:\n        return max(0, int(_env("TP_TOOL_TIMEOUT", "0")))\n    except ValueError:\n        return 0\n\n\ndef keep_notes_on_game_over() -> bool:\n    if not enabled():\n        return False\n    return _env("TP_KEEP_NOTES_ON_GAME_OVER", "1").lower() not in _OFF\n\n\ndef batch_cap() -> int:\n    """Requested actions per python tool call; 0 = unlimited."""\n    if not enabled():\n        return 0\n    try:\n        return max(0, int(_env("TP_BATCH_CAP", str(DEFAULT_BATCH_CAP))))\n    except ValueError:\n        return DEFAULT_BATCH_CAP\n\n\n# ------------------------------------------------------------- seam: trim ---\ndef _patch_trim(cls: Any) -> None:\n    stock_trim = cls._trim_messages_for_context\n\n    def trim(self, messages, *, tools=None, preserve_recent=1, extra_safety_tokens=0):\n        low = trim_low_water()\n        if low >= 1.0 or not messages:\n            return stock_trim(self, messages, tools=tools, preserve_recent=preserve_recent,\n                              extra_safety_tokens=extra_safety_tokens)\n        try:\n            system_message = messages[0]\n            history = list(messages[1:])\n            preserve_recent = max(0, preserve_recent)\n            budget = max(1, self._context_budget_tokens - max(0, extra_safety_tokens))\n            estimate = self._estimate_request_input_tokens([system_message, *history], tools=tools)\n            if estimate <= budget:\n                return [system_message, *self._drop_until_first_user_message(history)]\n            target = max(1, int(budget * low))\n            original = list(history)\n\n            def user_count(items):\n                return sum(1 for m in items if str(m.get("role", "")).strip() == "user")\n\n            while history and estimate > target:\n                # Never cut away the most recent user message: a request whose\n                # history is only assistant/tool turns is rejected by the\n                # server ("No user query found in messages", measured live).\n                if user_count(history) <= 1:\n                    break\n                if not self._drop_oldest_history_block(history, preserve_recent=preserve_recent):\n                    break\n                estimate = self._estimate_request_input_tokens([system_message, *history], tools=tools)\n            history = self._drop_until_first_user_message(history)\n            if not history and original:\n                # fall back to the stock trim rather than send an empty history\n                return stock_trim(self, messages, tools=tools, preserve_recent=preserve_recent,\n                                  extra_safety_tokens=extra_safety_tokens)\n            dropped = original[: max(0, len(original) - len(history))]\n            hook = ON_CUT\n            if hook is not None and dropped:\n                try:\n                    hook(self, dropped)\n                except Exception:  # noqa: BLE001 — a summary failure never blocks the turn\n                    pass\n            return [system_message, *history]\n        except Exception:  # noqa: BLE001 — fail open to stock\n            return stock_trim(self, messages, tools=tools, preserve_recent=preserve_recent,\n                              extra_safety_tokens=extra_safety_tokens)\n\n    trim._tp_stock = stock_trim\n    cls._trim_messages_for_context = trim\n\n\n# ------------------------------------------------------------- seam: init ---\ndef _patch_init(cls: Any) -> None:\n    stock_init = cls.__init__\n\n    def init(self, *args, **kwargs):\n        stock_init(self, *args, **kwargs)\n        try:\n            window = context_window()\n            if window > 0:\n                self._context_budget_tokens = max(\n                    1024, window - self._reply_reserve_tokens - self._request_safety_margin_tokens)\n            ys = yield_seconds()\n            if ys >= 0:\n                self._yield_seconds = None if ys == 0 else float(ys)\n            ts = tool_steps()\n            if ts >= 0:\n                self._tool_steps = None if ts == 0 else max(1, ts)\n            tt = tool_timeout()\n            if tt > 0:\n                # stock clamps LOCAL_ANALYZER_TOOL_TIMEOUT at 30 s; the measured\n                # cost on Flash-Next is ~130 guillotined turns per 11 games\n                # (R8 forensics) — actions execute but observations are lost.\n                self._python_timeout = max(1, tt)\n        except Exception:  # noqa: BLE001\n            pass\n\n    init._tp_stock = stock_init\n    cls.__init__ = init\n\n\n# ------------------------------------------------------------ seam: notes ---\ndef _patch_notes(cls: Any) -> None:\n    stock = cls._update_summarized_knowledge_from_step_summary\n\n    def update(self):\n        if not keep_notes_on_game_over():\n            return stock(self)\n        try:\n            summary = self._last_step_summary\n            if not summary:\n                return None\n            if summary.get("level_transition") or summary.get("run_complete"):\n                return stock(self)\n            return None  # game_over or nothing: keep every carried note\n        except Exception:  # noqa: BLE001\n            return stock(self)\n\n    update._tp_stock = stock\n    cls._update_summarized_knowledge_from_step_summary = update\n\n\n# -------------------------------------------------------- seam: batch cap ---\ndef begin_tool_call() -> None:\n    """Reset the per-tool-call action budget (called at every _run_python_tool)."""\n    _tls.remaining = batch_cap()\n\n\ndef _remaining() -> int | None:\n    cap = batch_cap()\n    if cap <= 0:\n        return None\n    remaining = getattr(_tls, "remaining", None)\n    if remaining is None:\n        remaining = cap\n        _tls.remaining = remaining\n    return remaining\n\n\ndef _patch_batch_cap(cls: Any) -> None:\n    stock_normalize = cls._normalize_python_actions\n    stock_run = cls._run_python_tool\n\n    def normalize(self, value):\n        normalized = stock_normalize(self, value)\n        try:\n            remaining = _remaining()\n        except Exception:  # noqa: BLE001\n            return normalized\n        if remaining is None:\n            return normalized\n        if remaining <= 0:\n            raise ValueError(BATCH_CAP_MESSAGE.format(cap=batch_cap()))\n        if len(normalized) > remaining:\n            normalized = normalized[:remaining]\n        _tls.remaining = remaining - len(normalized)\n        return normalized\n\n    def run(self, state_path, arguments):\n        begin_tool_call()\n        return stock_run(self, state_path, arguments)\n\n    normalize._tp_stock = stock_normalize\n    run._tp_stock = stock_run\n    cls._normalize_python_actions = normalize\n    cls._run_python_tool = run\n\n\n# ------------------------------------------------------------ time guard ---\ndef time_guard_per_game_s(stock_per_game_s: float, *, setup_elapsed_s: float, games: int,\n                          concurrency: int, total_budget_s: float = 32400.0,\n                          margin_s: float = 240.0) -> float:\n    """Shrink the per-game box only if setup + waves would overrun the 9 h box.\n\n    Never grows the box; never returns less than 600 s.\n    """\n    try:\n        waves = max(1, -(-int(games) // max(1, int(concurrency))))\n        fits = (float(total_budget_s) - float(setup_elapsed_s) - float(margin_s)) / waves\n        return float(min(float(stock_per_game_s), max(600.0, fits)))\n    except Exception:  # noqa: BLE001\n        return float(stock_per_game_s)\n\n\n# --------------------------------------------------------------- install ---\ndef install() -> str:\n    if _STATE["installed"]:\n        return "throughput: SKIP (already applied)"\n    try:\n        from inference.agent import tool_agent as agent_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"throughput: SKIP (tool_agent module missing: {exc!r})"\n    cls = getattr(agent_mod, "ToolAgent", None)\n    if cls is None:\n        return "throughput: SKIP (missing ToolAgent)"\n    for name in ("_trim_messages_for_context", "_update_summarized_knowledge_from_step_summary",\n                 "_normalize_python_actions", "_run_python_tool", "_estimate_request_input_tokens",\n                 "_drop_oldest_history_block", "_drop_until_first_user_message"):\n        if getattr(cls, name, None) is None:\n            return f"throughput: SKIP (ToolAgent.{name} missing)"\n    _patch_trim(cls)\n    _patch_init(cls)\n    _patch_notes(cls)\n    _patch_batch_cap(cls)\n    _STATE["installed"] = True\n    return "throughput: OK"\n\n\ndef status() -> dict[str, Any]:\n    return {\n        "installed": _STATE["installed"],\n        "enabled": enabled(),\n        "trim_low_water": trim_low_water(),\n        "context_window": context_window(),\n        "yield_seconds": yield_seconds(),\n        "tool_steps": tool_steps(),\n        "keep_notes_on_game_over": keep_notes_on_game_over(),\n        "tool_timeout": tool_timeout(),\n        "batch_cap": batch_cap(),\n    }\n', 'graft_control.py': '"""Memory & control graft (Pack 2, 2026-08-29) — the loop keeps its state,\nprobes before it reasons, and notices when it is stuck.\n\nInstalled AFTER graft_throughput (it relies on its ON_CUT hook and wraps the\nseams it already wrapped). Every behaviour has a TP2_* flag read at call time;\nTP2_ENABLE=0 makes every seam a pass-through.\n\nSeams:\n  (A) SUMMARY  graft_throughput.ON_CUT + ToolAgent._update_summarized_knowledge_from_step_summary\n               harness-owned summary: when the trimmer cuts history, a short\n               non-thinking completion compresses the dropped turns into the\n               seven carried note fields (the existing prompt channel); on a\n               level-up the whole history is compressed into cross-level notes\n               + action model before the level-specific keys are wiped.\n  (B) PROBE    _HarnessGameSession.play — at game start each available\n               keyboard action is executed once and up to TP2_PROBE_CLICKS\n               salient components are clicked; the effect table rides the\n               user prompt while the agent is on that level.\n  (C) STALL    _HarnessGameSession._execute_action + ToolAgent._build_user_prompt\n               + ToolAgent.analyze — HUD-aware frame hashing; after\n               TP2_STALL_T1 actions without a new board state a STAGNATION\n               directive is appended to the prompt; after TP2_STALL_T2 (30) the\n               harness issues a level RESET (max TP2_STALL_RESETS_PER_LEVEL).\n  (D) STREAK   _HarnessGameSession.step_env — inside one python tool call,\n               after TP2_STREAK_N consecutive no-effect actions further\n               actions are refused with a readable error.\n  (E) DIFF     _HarnessGameSession._execute_action + ToolAgent._compact_action_result\n               every action result carries a compact diff summary (changed\n               cells, HUD-excluded count, bbox, colours added/removed).\n\nEvidence: docs/STRATEGY-2026-08-29-independent-review-path-to-6.md §4,\ndocs/research-2026-08-29/R2-literature-mechanisms.md, R4 §1.2/§4.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport threading\nimport time\nfrom collections import Counter, deque\nfrom typing import Any\n\n_STATE = {"installed": False}\n_OFF = {"0", "false", "no", "off"}\n_lock = threading.Lock()\n# stock ToolAgent.analyze as captured at install (tests may swap it)\n_ANALYZE_STOCK: dict[str, Any] = {}\n\nSUMMARY_SYSTEM_PROMPT = (\n    "You compress the working notes of an agent playing an unknown 64x64 grid game. "\n    "From the transcript, write EXACTLY these seven lines and nothing else:\\n"\n    "World model: <what the level contains and how it behaves, verified facts first>\\n"\n    "Goal model: <what seems to complete the level; mark guesses as guesses>\\n"\n    "Action model: <what each action does, with exact effects>\\n"\n    "Recent findings: <the newest confirmed observations>\\n"\n    "Open questions: <what is still unknown>\\n"\n    "Plan: <the best next steps>\\n"\n    "Cross-level notes: <rules likely to hold on later levels>\\n"\n    "Keep coordinates, colours and counts exact. Keep ruled-out hypotheses as ruled out. "\n    "At most 120 words total; one line per field, no blank lines."\n)\nLEVEL_SUMMARY_SYSTEM_PROMPT = (\n    "The agent just completed a level of an unknown 64x64 grid game and moves to the next "\n    "level of the same game. From the transcript and notes, write EXACTLY these two lines:\\n"\n    "Action model: <what each action does, with exact effects; controls usually persist>\\n"\n    "Cross-level notes: <mechanics, goal pattern and lessons that should transfer; "\n    "drop layout details specific to the finished level>\\n"\n    "At most 150 words total."\n)\nSTAGNATION_T1_TEXT = (\n    "STAGNATION WARNING: the last {n} actions produced NO board state you had not already "\n    "seen on this level (HUD counters ignored). Do not repeat that pattern. Enumerate every "\n    "(action, target) you have NOT tried on this level — untested keyboard actions, unclicked "\n    "objects, different orderings — and take the most informative untried one now."\n)\nSTAGNATION_RESET_TEXT = (\n    "HARNESS NOTE: the level was RESET by the harness at action {at} after {n} actions without "\n    "a new board state; the board is back at the level\'s opening state. Your earlier notes still "\n    "apply. Choose a different approach than the one that stalled."\n)\nSTREAK_MESSAGE = (\n    "no_effect_streak: the last {n} actions changed nothing on the board. The rest of this batch "\n    "was not executed. Observe current_frame and pick a different action or object."\n)\n\n\n# ----------------------------------------------------------------- flags ---\ndef _env(name: str, default: str) -> str:\n    raw = os.environ.get(name)\n    return default if raw is None or not raw.strip() else raw.strip()\n\n\ndef _flag(name: str, default: str = "1") -> bool:\n    return _env(name, default).lower() not in _OFF\n\n\ndef _int(name: str, default: int) -> int:\n    try:\n        return int(_env(name, str(default)))\n    except ValueError:\n        return default\n\n\ndef enabled() -> bool:\n    return _flag("TP2_ENABLE")\n\n\ndef summary_enabled() -> bool:\n    return enabled() and _flag("TP2_SUMMARY")\n\n\ndef probe_enabled() -> bool:\n    return enabled() and _flag("TP2_PROBE")\n\n\ndef stall_enabled() -> bool:\n    return enabled() and _flag("TP2_STALL")\n\n\ndef streak_enabled() -> bool:\n    return enabled() and _flag("TP2_STREAK")\n\n\ndef diff_enabled() -> bool:\n    return enabled() and _flag("TP2_DIFF")\n\n\ndef probe_clicks() -> int:\n    return max(0, _int("TP2_PROBE_CLICKS", 3))\n\n\ndef stall_t1() -> int:\n    return max(1, _int("TP2_STALL_T1", 10))\n\n\ndef stall_t2() -> int:\n    return max(stall_t1() + 1, _int("TP2_STALL_T2", 30))\n\n\ndef stall_resets_per_level() -> int:\n    return max(0, _int("TP2_STALL_RESETS_PER_LEVEL", 2))\n\n\ndef streak_n() -> int:\n    return max(1, _int("TP2_STREAK_N", 3))\n\n\ndef summary_max_tokens() -> int:\n    return max(100, _int("TP2_SUMMARY_MAX_TOKENS", 300))\n\n\ndef summary_min_interval_s() -> float:\n    try:\n        return max(0.0, float(_env("TP2_SUMMARY_MIN_INTERVAL_S", "240")))\n    except ValueError:\n        return 240.0\n\n\ndef summary_min_chars() -> int:\n    return max(0, _int("TP2_SUMMARY_MIN_CHARS", 2000))\n\n\ndef summary_async() -> bool:\n    return _flag("TP2_SUMMARY_ASYNC", "1")\n\n\ndef status() -> dict[str, Any]:\n    return {\n        "installed": _STATE["installed"], "enabled": enabled(),\n        "summary": summary_enabled(), "probe": probe_enabled(), "probe_clicks": probe_clicks(),\n        "stall": stall_enabled(), "stall_t1": stall_t1(), "stall_t2": stall_t2(),\n        "streak": streak_enabled(), "streak_n": streak_n(), "diff": diff_enabled(),\n    }\n\n\n# ------------------------------------------------------------ primitives ---\nGrid = tuple[tuple[int, ...], ...]\n\n\ndef grid_hash(grid: Grid, mask: set[tuple[int, int]] | None = None) -> str:\n    h = hashlib.sha1()\n    for r, row in enumerate(grid):\n        if mask:\n            row = tuple(0 if (r, c) in mask else v for c, v in enumerate(row))\n        h.update(bytes(int(v) & 0xFF for v in row))\n        h.update(b"\\n")\n    return h.hexdigest()[:16]\n\n\nclass HudMask:\n    """Online HUD/timer detector: cells that change in most transitions."""\n\n    def __init__(self, min_transitions: int = 8, threshold: float = 0.6) -> None:\n        self.counts: Counter = Counter()\n        self.n = 0\n        self.min_transitions = min_transitions\n        self.threshold = threshold\n        self.size: tuple[int, int] = (64, 64)\n\n    def observe(self, before: Grid, after: Grid) -> None:\n        if not before or not after or len(before) != len(after):\n            return\n        self.size = (len(after), len(after[0]) if after[0] else 0)\n        changed = 0\n        for r, (rb, ra) in enumerate(zip(before, after)):\n            if rb == ra:\n                continue\n            for c, (vb, va) in enumerate(zip(rb, ra)):\n                if vb != va:\n                    self.counts[(r, c)] += 1\n                    changed += 1\n        if changed:\n            self.n += 1\n\n    def mask(self, edge_band: int | None = None, band_threshold: float = 0.3,\n             size: tuple[int, int] | None = None) -> set[tuple[int, int]] | None:\n        """Volatile cells, extended to whole edge-band rows/columns (HUD bars\n        and multi-digit counters live on the border; only their fastest digit\n        clears the per-cell threshold, so mask the bar, not the digit)."""\n        if self.n < self.min_transitions:\n            return None\n        cut = self.threshold * self.n\n        out = {cell for cell, cnt in self.counts.items() if cnt > cut}\n        h, w = size or self.size\n        if edge_band is None:\n            edge_band = max(1, h // 16)\n        row_sum: Counter = Counter()\n        col_sum: Counter = Counter()\n        for (r, c), cnt in self.counts.items():\n            row_sum[r] += cnt\n            col_sum[c] += cnt\n        band_rows = [r for r in range(h) if r < edge_band or r >= h - edge_band]\n        band_cols = [c for c in range(w) if c < edge_band or c >= w - edge_band]\n        for r in band_rows:\n            if any((r, c) in out for c in range(w)) or row_sum[r] / (self.n * w) >= band_threshold:\n                out.update((r, c) for c in range(w))\n        for c in band_cols:\n            if any((r, c) in out for r in range(h)) or col_sum[c] / (self.n * h) >= band_threshold:\n                out.update((r, c) for r in range(h))\n        return out or None\n\n\ndef _edge_only(diff: dict[str, Any] | None, edge_band: int | None = None,\n               size: tuple[int, int] = (64, 64)) -> bool:\n    """True when every changed non-HUD cell lies inside the border band."""\n    if not diff or not diff.get("bbox") or diff.get("changed_ex_hud", 0) == 0:\n        return False\n    r0, c0, r1, c1 = diff["bbox"]\n    h, w = size\n    if edge_band is None:\n        edge_band = max(1, h // 16)\n    rows_in_band = (r1 < edge_band) or (r0 >= h - edge_band)\n    cols_in_band = (c1 < edge_band) or (c0 >= w - edge_band)\n    return rows_in_band or cols_in_band\n\n\ndef diff_summary(before: Grid, after: Grid, mask: set[tuple[int, int]] | None = None) -> dict[str, Any]:\n    cells: list[tuple[int, int]] = []\n    added: Counter = Counter()\n    removed: Counter = Counter()\n    if before and after and len(before) == len(after):\n        for r, (rb, ra) in enumerate(zip(before, after)):\n            if rb == ra:\n                continue\n            for c, (vb, va) in enumerate(zip(rb, ra)):\n                if vb != va:\n                    cells.append((r, c))\n                    added[int(va)] += 1\n                    removed[int(vb)] += 1\n    ex = [cell for cell in cells if not mask or cell not in mask]\n    box = ex or cells\n    bbox = [min(r for r, _ in box), min(c for _, c in box), max(r for r, _ in box), max(c for _, c in box)] if box else None\n    return {\n        "changed": len(cells),\n        "changed_ex_hud": len(ex),\n        "bbox": bbox,\n        "colors_added": sorted(added),\n        "colors_removed": sorted(removed),\n    }\n\n\ndef components(grid: Grid, background: int | None = None) -> list[dict[str, Any]]:\n    if not grid:\n        return []\n    h, w = len(grid), len(grid[0])\n    if background is None:\n        background = Counter(v for row in grid for v in row).most_common(1)[0][0]\n    seen = [[False] * w for _ in range(h)]\n    out: list[dict[str, Any]] = []\n    for r0 in range(h):\n        for c0 in range(w):\n            if seen[r0][c0] or grid[r0][c0] == background:\n                continue\n            color = grid[r0][c0]\n            stack = [(r0, c0)]\n            seen[r0][c0] = True\n            cells = []\n            while stack:\n                r, c = stack.pop()\n                cells.append((r, c))\n                for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):\n                    if 0 <= nr < h and 0 <= nc < w and not seen[nr][nc] and grid[nr][nc] == color:\n                        seen[nr][nc] = True\n                        stack.append((nr, nc))\n            rs = [r for r, _ in cells]\n            cs = [c for _, c in cells]\n            out.append({\n                "color": int(color), "area": len(cells), "cells": cells,\n                "bbox": [min(rs), min(cs), max(rs), max(cs)],\n                "center": (round(sum(rs) / len(rs)), round(sum(cs) / len(cs))),\n            })\n    out.sort(key=lambda d: d["area"])\n    return out\n\n\ndef salient_clicks(grid: Grid, k: int) -> list[tuple[int, int, dict[str, Any]]]:\n    """Centres of up to k small, rare-colour, non-background components."""\n    comps = components(grid)\n    if not comps or k <= 0:\n        return []\n    color_area: Counter = Counter()\n    for comp in comps:\n        color_area[comp["color"]] += comp["area"]\n\n    def score(comp: dict[str, Any]) -> tuple:\n        small = 0 if 2 <= comp["area"] <= 60 else 1\n        return (small, color_area[comp["color"]], comp["area"])\n\n    picked: list[tuple[int, int, dict[str, Any]]] = []\n    used_colors: Counter = Counter()\n    for comp in sorted(comps, key=score):\n        if used_colors[comp["color"]] >= 2:\n            continue\n        r, c = comp["center"]\n        if (r, c) not in set(comp["cells"]):\n            r, c = min(comp["cells"], key=lambda rc: abs(rc[0] - r) + abs(rc[1] - c))\n        picked.append((r, c, {"color": comp["color"], "area": comp["area"], "bbox": comp["bbox"]}))\n        used_colors[comp["color"]] += 1\n        if len(picked) >= k:\n            break\n    return picked\n\n\n# ------------------------------------------------------- session state ---\nclass SessionState:\n    def __init__(self) -> None:\n        self.hud = HudMask()\n        self.seen: set[str] = set()\n        self.since_new = 0\n        self.level: int | None = None\n        self.resets_this_level = 0\n        self.last_reset_note: str | None = None\n        self.streak = 0\n        self.actions = 0\n        self.recent_hashes: deque = deque(maxlen=64)\n\n\ndef _state(session: Any) -> SessionState:\n    st = getattr(session, "_tp2", None)\n    if st is None:\n        st = SessionState()\n        try:\n            session._tp2 = st\n        except Exception:  # noqa: BLE001\n            pass\n    return st\n\n\ndef _reset_available(session: Any, arcengine: Any) -> bool:\n    """RESET is filtered out of the model-facing valid_actions by the solver;\n    ask the engine state directly (absent list => assume available)."""\n    try:\n        available = session.game.current_state.available_actions\n    except Exception:  # noqa: BLE001\n        return True\n    try:\n        ids = set(int(a) for a in (available or []))\n    except Exception:  # noqa: BLE001\n        return True\n    return (not ids) or int(arcengine.GameAction.RESET.value) in ids\n\n\ndef _session_of(agent: Any) -> Any:\n    cb = getattr(agent, "_step_env_callback", None)\n    return getattr(cb, "__self__", None)\n\n\ndef _grid(session: Any, solver_mod: Any) -> Grid:\n    try:\n        return solver_mod._grid_from_state(session.game.current_state)\n    except Exception:  # noqa: BLE001\n        return ()\n\n\ndef _after_action(st: SessionState, before: Grid, after: Grid, payload: dict[str, Any]) -> None:\n    """Update diff / HUD / stall / streak state after one executed action."""\n    mask = st.hud.mask()\n    diff = diff_summary(before, after, mask) if (before and after) else None\n    if diff is not None and diff_enabled():\n        payload["diff"] = diff\n    st.hud.observe(before, after)\n    st.actions += 1\n    level = payload.get("level")\n    if level is not None and level != st.level:\n        st.level = level\n        st.seen = set()\n        st.since_new = 0\n        st.resets_this_level = 0\n    if after:\n        h = grid_hash(after, mask)\n        st.recent_hashes.append(h)\n        if h in st.seen:\n            st.since_new += 1\n        elif diff is not None and (diff["changed_ex_hud"] == 0\n                                   or _edge_only(diff, size=(len(after), len(after[0]) if after[0] else 0))):\n            # only HUD/border cells moved: not a new gameplay state\n            st.seen.add(h)\n            st.since_new += 1\n        else:\n            st.seen.add(h)\n            st.since_new = 0\n    if payload.get("executed"):\n        no_effect = (diff["changed_ex_hud"] == 0) if diff is not None else (not payload.get("board_changed"))\n        animated = int(payload.get("frame_count") or 1) > 1\n        st.streak = st.streak + 1 if (no_effect and not animated) else 0\n\n\n# ------------------------------------------------------------ summaries ---\ndef _transcript(messages: list[dict[str, Any]], cap: int = 14000) -> str:\n    parts: list[str] = []\n    for m in messages:\n        role = str(m.get("role", ""))\n        if role == "assistant":\n            content = m.get("content")\n            if isinstance(content, str) and content.strip():\n                parts.append("ASSISTANT: " + content.strip()[:1500])\n            for call in m.get("tool_calls") or []:\n                fn = call.get("function", {}) if isinstance(call, dict) else {}\n                args = fn.get("arguments", "")\n                if isinstance(args, dict):\n                    args = json.dumps(args)\n                parts.append("TOOL CALL: " + str(args)[:600])\n        elif role == "tool":\n            parts.append("TOOL RESULT: " + str(m.get("content", ""))[:700])\n        elif role == "user":\n            content = m.get("content")\n            if isinstance(content, list):\n                content = " ".join(str(p.get("text", "")) for p in content if isinstance(p, dict))\n            text = str(content or "")\n            head = text.split("\\n", 2)[:2]\n            parts.append("USER: " + " ".join(head)[:300])\n    text = "\\n".join(parts)\n    return text[-cap:] if len(text) > cap else text\n\n\ndef _post_summary(agent: Any, system_prompt: str, user_text: str, agent_mod: Any) -> str:\n    import requests  # noqa: PLC0415 — the bundle already depends on it\n\n    from inference.utils.openai_compat import build_chat_payload  # noqa: PLC0415\n\n    model = agent._model\n    payload = build_chat_payload(\n        provider=model.provider, model=model.model_id,\n        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_text}],\n        max_tokens=summary_max_tokens(), temperature=0.2, top_p=0.95, top_k=20,\n        thinking=False, tools=None, tool_choice=None, seed=None,\n    )\n    response = requests.post(f"{model.base_url.rstrip(\'/\')}/chat/completions",\n                             headers=agent._headers(), json=payload, timeout=180)\n    response.raise_for_status()\n    message = response.json()["choices"][0]["message"]\n    content = message.get("content") or ""\n    if isinstance(content, list):\n        content = " ".join(str(p.get("text", "")) for p in content if isinstance(p, dict))\n    return str(content)\n\n\ndef _notes_text(agent: Any) -> str:\n    notes = getattr(agent, "_summarized_knowledge", {}) or {}\n    labels = [("World model", "world_model"), ("Goal model", "goal_model"), ("Action model", "action_model"),\n              ("Recent findings", "recent_findings"), ("Open questions", "open_questions"),\n              ("Plan", "current_plan"), ("Cross-level notes", "cross_level_notes")]\n    lines = [f"{label}: {notes.get(key)}" for label, key in labels if notes.get(key)]\n    return "\\n".join(lines)\n\n\ndef _summary_allowed(agent: Any, text: str) -> bool:\n    """Rate limit: skip tiny cuts and cuts closer than the minimum interval."""\n    if len(text) < summary_min_chars():\n        return False\n    now = time.monotonic()\n    last = getattr(agent, "_tp2_last_summary_at", None)\n    if last is not None and (now - last) < summary_min_interval_s():\n        return False\n    try:\n        agent._tp2_last_summary_at = now\n    except Exception:  # noqa: BLE001\n        pass\n    return True\n\n\ndef _summarize_into_notes(agent: Any, dropped: list[dict[str, Any]], agent_mod: Any,\n                          *, force: bool = False) -> bool:\n    text = _transcript(dropped)\n    if not text.strip():\n        return False\n    if not force and not _summary_allowed(agent, text):\n        return False\n    prev = _notes_text(agent)\n    user_text = ("Previous notes:\\n" + prev + "\\n\\n" if prev else "") + "Transcript of the turns being compressed:\\n" + text\n    content = _post_summary(agent, SUMMARY_SYSTEM_PROMPT, user_text, agent_mod)\n    note = agent_mod._extract_scientist_note(content)\n    if not note or not any(note.values()):\n        return False\n    with _lock:\n        for key, value in note.items():\n            if value:\n                agent._summarized_knowledge[key] = value\n    return True\n\n\ndef _summarize_level_boundary(agent: Any, agent_mod: Any) -> dict[str, str]:\n    history = list(getattr(agent, "_history_messages", []) or [])\n    text = _transcript(history)\n    prev = _notes_text(agent)\n    user_text = ("Notes so far:\\n" + prev + "\\n\\n" if prev else "") + "Transcript:\\n" + text\n    content = _post_summary(agent, LEVEL_SUMMARY_SYSTEM_PROMPT, user_text, agent_mod)\n    note = agent_mod._extract_scientist_note(content)\n    return {k: v for k, v in (note or {}).items() if v and k in ("action_model", "cross_level_notes")}\n\n\n# --------------------------------------------------------------- probe ---\n_KEYBOARD = [("ACTION1", "UP"), ("ACTION2", "DOWN"), ("ACTION3", "LEFT"), ("ACTION4", "RIGHT"), ("ACTION5", "SPACE")]\n\n\ndef _fmt_effect(diff: dict[str, Any] | None, payload: dict[str, Any]) -> str:\n    if payload.get("level_completed"):\n        return "COMPLETED THE LEVEL"\n    if payload.get("game_over"):\n        return "GAME OVER (level was reset)"\n    if not diff:\n        return "board changed" if payload.get("board_changed") else "no effect"\n    if diff["changed_ex_hud"] == 0 and diff["changed"] == 0:\n        return "no effect"\n    if diff["changed_ex_hud"] == 0:\n        return f"only HUD-like cells changed ({diff[\'changed\']})"\n    b = diff["bbox"]\n    return (f"changed {diff[\'changed_ex_hud\']} cells in rows {b[0]}-{b[2]} cols {b[1]}-{b[3]}; "\n            f"colours appeared {diff[\'colors_added\']} vanished {diff[\'colors_removed\']}")\n\n\ndef run_probe(session: Any, solver_mod: Any, arcengine: Any) -> dict[str, Any] | None:\n    """Execute the level-start probe on a fresh game. Returns the table record."""\n    game = session.game\n    state = game.current_state\n    try:\n        available = set(int(a) for a in state.available_actions)\n    except Exception:  # noqa: BLE001\n        return None\n    st = _state(session)\n    try:\n        level = int(solver_mod._level_number(game))\n    except Exception:  # noqa: BLE001\n        level = int(getattr(state, "levels_completed", 0) or 0) + 1\n    lines: list[str] = []\n    executed = 0\n\n    def do(action_name: str, data: dict[str, Any], label: str) -> dict[str, Any]:\n        nonlocal executed\n        action = arcengine.ActionInput(id=arcengine.GameAction.from_name(action_name), data=data)\n        before = _grid(session, solver_mod)\n        payload = session._execute_action(action, batch_index=1, batch_size=1, generated_tokens=0)\n        executed += 1\n        diff = payload.get("diff")\n        if diff is None:\n            after = _grid(session, solver_mod)\n            diff = diff_summary(before, after, st.hud.mask()) if before and after else None\n        lines.append(f"- {label}: {_fmt_effect(diff, payload)}")\n        return payload\n\n    stop = False\n    for engine_name, label in _KEYBOARD:\n        value = int(arcengine.GameAction.from_name(engine_name).value)\n        if value not in available:\n            continue\n        payload = do(engine_name, {}, label)\n        if payload.get("level_completed") or payload.get("run_complete") or payload.get("game_over"):\n            stop = True\n            break\n        try:\n            available = set(int(a) for a in game.current_state.available_actions)\n        except Exception:  # noqa: BLE001\n            pass\n    click_value = int(arcengine.GameAction.from_name("ACTION6").value)\n    if not stop and click_value in available and probe_clicks() > 0:\n        grid = _grid(session, solver_mod)\n        for r, c, info in salient_clicks(grid, probe_clicks()):\n            payload = do("ACTION6", {"x": int(c), "y": int(r)},\n                         f"MOUSE(row {r}, col {c}) on a colour-{info[\'color\']} object of {info[\'area\']} cells")\n            if payload.get("level_completed") or payload.get("run_complete") or payload.get("game_over"):\n                break\n    if not lines:\n        return None\n    text = ("Harness probe at the start of this level (each available action tried once; "\n            f"{executed} actions spent, all recorded in `history`):\\n" + "\\n".join(lines))\n    return {"level": level, "text": text, "actions": executed}\n\n\n# --------------------------------------------------------------- install ---\ndef install() -> str:\n    if _STATE["installed"]:\n        return "control: SKIP (already applied)"\n    try:\n        import graft_throughput as tp  # noqa: PLC0415\n    except Exception as exc:  # noqa: BLE001\n        return f"control: SKIP (graft_throughput missing: {exc!r})"\n    if not tp._STATE.get("installed"):\n        return "control: SKIP (graft_throughput not installed)"\n    try:\n        import arcengine  # noqa: PLC0415\n        from inference.agent import tool_agent as agent_mod  # noqa: PLC0415\n        from inference.framework import solver as solver_mod  # noqa: PLC0415\n    except Exception as exc:  # noqa: BLE001\n        return f"control: SKIP (import failed: {exc!r})"\n    agent_cls = agent_mod.ToolAgent\n    session_cls = solver_mod._HarnessGameSession\n    for name in ("_build_user_prompt", "analyze", "_compact_action_result", "_run_python_tool",\n                 "_update_summarized_knowledge_from_step_summary"):\n        if getattr(agent_cls, name, None) is None:\n            return f"control: SKIP (ToolAgent.{name} missing)"\n    for name in ("play", "step_env", "_execute_action", "_error_payload"):\n        if getattr(session_cls, name, None) is None:\n            return f"control: SKIP (_HarnessGameSession.{name} missing)"\n\n    # (E)+(C)+(D) state: _execute_action ---------------------------------\n    stock_execute_action = session_cls._execute_action\n\n    def execute_action_wrapped(self, action, *args, **kwargs):\n        before = _grid(self, solver_mod) if enabled() else ()\n        payload = stock_execute_action(self, action, *args, **kwargs)\n        if not enabled():\n            return payload\n        try:\n            _after_action(_state(self), before, _grid(self, solver_mod), payload)\n        except Exception:  # noqa: BLE001\n            pass\n        return payload\n\n    execute_action_wrapped._tp2_stock = stock_execute_action\n    session_cls._execute_action = execute_action_wrapped\n\n    stock_aggregate = agent_mod._aggregate_action_batch_result\n\n    def aggregate(*args, **kwargs):\n        out = stock_aggregate(*args, **kwargs)\n        try:\n            if diff_enabled():\n                executed = kwargs.get("executed_results") if "executed_results" in kwargs else (args[1] if len(args) > 1 else [])\n                diffs = [r.get("diff") for r in (executed or []) if isinstance(r, dict) and isinstance(r.get("diff"), dict)]\n                if diffs:\n                    last = dict(diffs[-1])\n                    last["batch_changed_ex_hud_total"] = sum(int(d.get("changed_ex_hud", 0)) for d in diffs)\n                    out["diff"] = last\n        except Exception:  # noqa: BLE001\n            pass\n        return out\n\n    aggregate._tp2_stock = stock_aggregate\n    agent_mod._aggregate_action_batch_result = aggregate\n\n    stock_compact = agent_cls._compact_action_result\n\n    def compact(self, payload):\n        out = stock_compact(self, payload)\n        try:\n            if diff_enabled() and isinstance(payload, dict) and isinstance(payload.get("diff"), dict):\n                out["diff"] = dict(payload["diff"])\n        except Exception:  # noqa: BLE001\n            pass\n        return out\n\n    compact._tp2_stock = stock_compact\n    agent_cls._compact_action_result = compact\n\n    # (D) streak halt: step_env + reset at each python tool call ----------\n    stock_step = session_cls.step_env\n\n    def step_env(self, arguments):\n        try:\n            if streak_enabled() and not (isinstance(arguments, dict) and arguments.get("query")):\n                st = _state(self)\n                if st.streak >= streak_n():\n                    return self._error_payload(STREAK_MESSAGE.format(n=st.streak))\n        except Exception:  # noqa: BLE001\n            pass\n        return stock_step(self, arguments)\n\n    step_env._tp2_stock = stock_step\n    session_cls.step_env = step_env\n\n    stock_run = agent_cls._run_python_tool\n\n    def run_tool(self, state_path, arguments):\n        try:\n            sess = _session_of(self)\n            if sess is not None:\n                _state(sess).streak = 0\n        except Exception:  # noqa: BLE001\n            pass\n        return stock_run(self, state_path, arguments)\n\n    run_tool._tp2_stock = stock_run\n    agent_cls._run_python_tool = run_tool\n\n    # (B)+(C) prompt: probe table + stagnation directive -----------------\n    stock_prompt = agent_cls._build_user_prompt\n\n    def build_prompt(self, action_num, **kwargs):\n        text = stock_prompt(self, action_num, **kwargs)\n        if not enabled():\n            return text\n        try:\n            extra: list[str] = []\n            current_frame = kwargs.get("current_frame")\n            level = getattr(current_frame, "level", None)\n            probe = getattr(self, "_tp2_probe", None)\n            if probe and probe_enabled() and (level is None or int(level) == int(probe["level"])):\n                extra.append(probe["text"])\n            sess = _session_of(self)\n            if sess is not None and stall_enabled():\n                st = _state(sess)\n                if st.last_reset_note:\n                    extra.append(st.last_reset_note)\n                    st.last_reset_note = None\n                if st.since_new >= stall_t1():\n                    extra.append(STAGNATION_T1_TEXT.format(n=st.since_new))\n            if extra:\n                text = text + "\\n" + "\\n".join(extra)\n        except Exception:  # noqa: BLE001\n            pass\n        return text\n\n    build_prompt._tp2_stock = stock_prompt\n    agent_cls._build_user_prompt = build_prompt\n\n    stock_analyze = agent_cls.analyze\n\n    def analyze(self, state_path, action_num, valid_actions=None, step_env=None, **kwargs):\n        try:\n            sess = getattr(step_env, "__self__", None)\n            if sess is not None and stall_enabled():\n                st = _state(sess)\n                if st.since_new >= stall_t2() and st.resets_this_level < stall_resets_per_level() \\\n                        and _reset_available(sess, arcengine):\n                    action = arcengine.ActionInput(id=arcengine.GameAction.RESET, data={})\n                    sess._execute_action(action, batch_index=1, batch_size=1, generated_tokens=0)\n                    st.resets_this_level += 1\n                    st.last_reset_note = STAGNATION_RESET_TEXT.format(at=sess.action_count, n=st.since_new)\n                    st.since_new = 0\n                    try:\n                        sess.write_runtime_state()\n                    except Exception:  # noqa: BLE001\n                        pass\n                    action_num = sess.action_count\n                    valid_actions = solver_mod._engine_action_names(sess.game)\n        except Exception:  # noqa: BLE001\n            pass\n        return _ANALYZE_STOCK["fn"](self, state_path, action_num, valid_actions=valid_actions,\n                                    step_env=step_env, **kwargs)\n\n    _ANALYZE_STOCK["fn"] = stock_analyze\n    analyze._tp2_stock = stock_analyze\n    agent_cls.analyze = analyze\n\n    # (B) probe at game start ---------------------------------------------\n    stock_play = session_cls.play\n\n    def play(self):\n        if probe_enabled():\n            try:\n                if int(getattr(self, "action_count", 0) or 0) == 0:\n                    self.seed_initial_history()\n                    record = run_probe(self, solver_mod, arcengine)\n                    if record:\n                        self.analyzer._tp2_probe = record\n                        self.write_runtime_state()\n            except Exception:  # noqa: BLE001\n                pass\n        return stock_play(self)\n\n    play._tp2_stock = stock_play\n    session_cls.play = play\n\n    # (A) summaries -------------------------------------------------------\n    def on_cut(agent, dropped):\n        if not summary_enabled():\n            return\n        if not summary_async():\n            _summarize_into_notes(agent, dropped, agent_mod)\n            return\n        # Rate-limit on the caller\'s thread, then run the summary call in the\n        # background so the turn is never blocked; the merge lands under a\n        # lock and the next prompt build picks it up (measured: a blocking\n        # 600-token summary cost ~70 s per cut at concurrency 28).\n        text = _transcript(list(dropped))\n        if not _summary_allowed(agent, text):\n            return\n        if getattr(agent, "_tp2_summary_inflight", False):\n            return\n        agent._tp2_summary_inflight = True\n        snapshot = list(dropped)\n\n        def worker():\n            try:\n                _summarize_into_notes(agent, snapshot, agent_mod, force=True)\n            except Exception:  # noqa: BLE001\n                pass\n            finally:\n                agent._tp2_summary_inflight = False\n\n        threading.Thread(target=worker, name="tp2-summary", daemon=True).start()\n\n    tp.ON_CUT = on_cut\n\n    stock_notes = agent_cls._update_summarized_knowledge_from_step_summary\n\n    def update_notes(self):\n        try:\n            summary = self._last_step_summary\n            if summary_enabled() and summary and summary.get("level_transition"):\n                carried = _summarize_level_boundary(self, agent_mod)\n                result = stock_notes(self)\n                for key, value in carried.items():\n                    self._summarized_knowledge[key] = value\n                return result\n        except Exception:  # noqa: BLE001\n            pass\n        return stock_notes(self)\n\n    update_notes._tp2_stock = stock_notes\n    agent_cls._update_summarized_knowledge_from_step_summary = update_notes\n\n    _STATE["installed"] = True\n    return "control: OK"\n', 'frontier_explorer.py': '"""Frontier explorer — a model-free level explorer (Pack 4, 2026-08-29).\n\nA compact port of the "just-explore" method (dolphin-in-a-coma/arc-agi-3-\njust-explore, MIT; 3rd in the ARC-AGI-3 preview with zero LLM calls; median\n17 private levels after the graph-reset fix). The idea:\n\n  * Frame processing: 4-connected single-colour segments; status bars are\n    segments hugging a border (aspect ratio >= 5, or >= 3 same-shaped twins on\n    the same border) and are masked before hashing; click candidates are one\n    per segment, grouped into five priority tiers (salient colour + medium\n    size first, status-bar segments last); keyboard actions sit in tier 0.\n  * Level graph: nodes are masked-frame hashes; each node keeps its untested\n    candidates; the explorer takes an untested candidate in the active tier\n    at the current node, or walks the shortest path to the nearest node that\n    still has one (frontier); when nothing is reachable it opens the next tier.\n\nPure Python, no numpy; a 64x64 frame segments in a few milliseconds.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport random\nfrom collections import Counter, deque\nfrom dataclasses import dataclass, field\nfrom typing import Any\n\nGrid = tuple[tuple[int, ...], ...]\n\nSALIENT = set(range(6, 16))\nMIN_WIDTH, MAX_WIDTH = 2, 32\nEDGE_DIST = 3\nBAR_RATIO = 5\nTWINS = 3\nN_GROUPS = 5\nKEYBOARD = {1: "ACTION1", 2: "ACTION2", 3: "ACTION3", 4: "ACTION4", 5: "ACTION5"}\n\n\n# ---------------------------------------------------------------- frames ---\ndef segments(grid: Grid) -> list[dict[str, Any]]:\n    """4-connected same-colour components over the whole grid (background too)."""\n    if not grid:\n        return []\n    h, w = len(grid), len(grid[0])\n    label = [[-1] * w for _ in range(h)]\n    out: list[dict[str, Any]] = []\n    for r0 in range(h):\n        for c0 in range(w):\n            if label[r0][c0] >= 0:\n                continue\n            color = grid[r0][c0]\n            idx = len(out)\n            label[r0][c0] = idx\n            stack = [(r0, c0)]\n            cells = []\n            while stack:\n                r, c = stack.pop()\n                cells.append((r, c))\n                for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):\n                    if 0 <= nr < h and 0 <= nc < w and label[nr][nc] < 0 and grid[nr][nc] == color:\n                        label[nr][nc] = idx\n                        stack.append((nr, nc))\n            rs = [r for r, _ in cells]\n            cs = [c for _, c in cells]\n            r_min, r_max, c_min, c_max = min(rs), max(rs), min(cs), max(cs)\n            out.append({\n                "id": idx, "color": int(color), "area": len(cells), "cells": cells,\n                "bbox": (r_min, c_min, r_max, c_max),\n                "height": r_max - r_min + 1, "width": c_max - c_min + 1,\n                "rect": len(cells) == (r_max - r_min + 1) * (c_max - c_min + 1),\n            })\n    return out\n\n\ndef _edges_of(seg: dict[str, Any], h: int, w: int) -> list[str]:\n    r_min, c_min, r_max, c_max = seg["bbox"]\n    edges = []\n    if c_max < EDGE_DIST:\n        edges.append("left")\n    if c_min > w - 1 - EDGE_DIST:\n        edges.append("right")\n    if r_max < EDGE_DIST:\n        edges.append("top")\n    if r_min > h - 1 - EDGE_DIST:\n        edges.append("bottom")\n    return edges\n\n\ndef status_bar_mask(grid: Grid, segs: list[dict[str, Any]]) -> set[tuple[int, int]]:\n    """just-explore\'s rule: border-hugging bars (aspect >= 5) or >= 3 twins on a border."""\n    if not grid:\n        return set()\n    h, w = len(grid), len(grid[0])\n    mask: set[tuple[int, int]] = set()\n    by_edge: dict[str, list[dict[str, Any]]] = {}\n    for seg in segs:\n        for e in _edges_of(seg, h, w):\n            by_edge.setdefault(e, []).append(seg)\n    for edge, group in by_edge.items():\n        horizontal = edge in ("top", "bottom")\n        for seg in group:\n            ratio = seg["width"] / seg["height"] if seg["height"] else 0\n            is_bar = (ratio >= BAR_RATIO) if horizontal else (ratio <= 1 / BAR_RATIO if ratio else False)\n            twins = [t for t in group if t is not seg and t["color"] == seg["color"]\n                     and t["rect"] == seg["rect"] and t["width"] == seg["width"] and t["height"] == seg["height"]]\n            if is_bar or len(twins) + 1 >= TWINS:\n                mask.update(seg["cells"])\n    return mask\n\n\ndef frame_hash(grid: Grid, mask: set[tuple[int, int]]) -> str:\n    h = hashlib.sha1()\n    for r, row in enumerate(grid):\n        vals = bytes((16 if (r, c) in mask else int(v)) & 0xFF for c, v in enumerate(row))\n        h.update(vals)\n        h.update(b"\\n")\n    return h.hexdigest()[:20]\n\n\ndef group_of(seg: dict[str, Any], masked: bool) -> int:\n    salient = seg["color"] in SALIENT\n    medium = MIN_WIDTH <= seg["width"] <= MAX_WIDTH and MIN_WIDTH <= seg["height"] <= MAX_WIDTH\n    if masked:\n        return 4\n    if salient and medium:\n        return 0\n    if medium:\n        return 1\n    if salient:\n        return 2\n    return 3\n\n\n@dataclass\nclass Candidate:\n    kind: str                 # "key" or "click"\n    action: str               # engine action name\n    data: dict[str, Any]      # {} or {"x":..,"y":..}\n    group: int\n    label: str\n\n\ndef candidates_for(grid: Grid, available: set[int], mask: set[tuple[int, int]],\n                   segs: list[dict[str, Any]], rng: random.Random) -> list[Candidate]:\n    out: list[Candidate] = []\n    for value, name in KEYBOARD.items():\n        if value in available:\n            out.append(Candidate("key", name, {}, 0, name))\n    if 6 in available:\n        for seg in segs:\n            cells = seg["cells"]\n            masked = all(cell in mask for cell in cells[: min(len(cells), 8)])\n            r, c = cells[rng.randrange(len(cells))]\n            out.append(Candidate("click", "ACTION6", {"x": int(c), "y": int(r)}, group_of(seg, masked),\n                                 f"click({r},{c}) colour {seg[\'color\']} area {seg[\'area\']}"))\n    return out\n\n\n# ----------------------------------------------------------------- graph ---\n@dataclass\nclass Node:\n    key: str\n    candidates: list[Candidate]\n    untested: dict[int, set[int]] = field(default_factory=dict)   # group -> candidate indices\n    result: dict[int, int] = field(default_factory=dict)          # idx -> 1 changed / -1 no change\n    target: dict[int, str] = field(default_factory=dict)          # idx -> node key\n\n    def __post_init__(self) -> None:\n        for i, cand in enumerate(self.candidates):\n            self.untested.setdefault(cand.group, set()).add(i)\n\n    def open_in(self, active: int) -> list[int]:\n        return [i for g in range(active + 1) for i in sorted(self.untested.get(g, ()))]\n\n\nclass FrontierExplorer:\n    def __init__(self, seed: int = 0) -> None:\n        self.rng = random.Random(seed)\n        self.reset()\n\n    def reset(self) -> None:\n        self.nodes: dict[str, Node] = {}\n        self.rev: dict[str, set[tuple[str, int]]] = {}     # target -> {(source, idx)}\n        self.active = 0\n        self.dist: dict[str, int] = {}\n        self.next_hop: dict[str, int] = {}                 # node -> candidate idx towards frontier\n        self.last: tuple[str, int] | None = None\n        self.mask: set[tuple[int, int]] | None = None\n        self.stats = Counter()\n\n    # -- observation --------------------------------------------------------\n    def observe(self, grid: Grid, available: list[int] | set[int]) -> str:\n        segs = segments(grid)\n        if self.mask is None:\n            self.mask = status_bar_mask(grid, segs)\n        key = frame_hash(grid, self.mask)\n        if key not in self.nodes:\n            cands = candidates_for(grid, set(int(a) for a in available), self.mask, segs, self.rng)\n            self.nodes[key] = Node(key, cands)\n            self.stats["nodes"] += 1\n            self._rebuild()\n        return key\n\n    # -- learning -----------------------------------------------------------\n    def record(self, prev_key: str, idx: int, new_key: str) -> None:\n        node = self.nodes.get(prev_key)\n        if node is None or idx not in range(len(node.candidates)):\n            return\n        changed = new_key != prev_key\n        node.result[idx] = 1 if changed else -1\n        node.untested.get(node.candidates[idx].group, set()).discard(idx)\n        if changed:\n            node.target[idx] = new_key\n            self.rev.setdefault(new_key, set()).add((prev_key, idx))\n            self.stats["edges"] += 1\n        else:\n            self.stats["noops"] += 1\n        self._rebuild()\n\n    # -- decision -----------------------------------------------------------\n    def choose(self, key: str) -> tuple[int, str]:\n        node = self.nodes[key]\n        while True:\n            open_idx = node.open_in(self.active)\n            if open_idx:\n                # lowest group first, random inside the group\n                best_group = min(node.candidates[i].group for i in open_idx)\n                pool = [i for i in open_idx if node.candidates[i].group == best_group]\n                i = self.rng.choice(pool)\n                self.stats["explore"] += 1\n                return i, f"untested tier {best_group}"\n            hop = self.next_hop.get(key)\n            if hop is not None:\n                self.stats["travel"] += 1\n                return hop, f"towards frontier (dist {self.dist.get(key)})"\n            if self.active < N_GROUPS - 1:\n                self.active += 1\n                self.stats["tier_advance"] += 1\n                self._rebuild()\n                continue\n            # everything exhausted: random tested-changing edge, else random candidate\n            changing = [i for i, r in node.result.items() if r == 1]\n            i = self.rng.choice(changing) if changing else self.rng.randrange(len(node.candidates))\n            self.stats["random"] += 1\n            return i, "exhausted: random"\n\n    # -- distances (BFS from frontier over reverse edges) --------------------\n    def _rebuild(self) -> None:\n        frontier = [k for k, n in self.nodes.items() if n.open_in(self.active)]\n        dist: dict[str, int] = {k: 0 for k in frontier}\n        hop: dict[str, int] = {}\n        dq = deque(frontier)\n        while dq:\n            cur = dq.popleft()\n            for src, idx in self.rev.get(cur, ()):\n                if src not in dist:\n                    dist[src] = dist[cur] + 1\n                    hop[src] = idx\n                    dq.append(src)\n        self.dist, self.next_hop = dist, hop\n        self.stats["frontier"] = len(frontier)\n\n    def frontier_size(self) -> int:\n        return sum(1 for n in self.nodes.values() if n.open_in(self.active))\n', 'graft_explore.py': '"""Explorer fallback graft (Pack 4, 2026-08-29) — a model-free frontier walk\ntakes over a stuck level, or the last minutes of a game, and hands back.\n\nInstalled AFTER graft_control. Two triggers, both checked at the top of every\nLLM turn (ToolAgent.analyze) from the session state Pack 2 maintains:\n\n  STALL TIER 3   after the harness RESET tier has fired at least once on this\n                 level and since_new >= TP4_STALL_T3 (default 30) again, and\n                 the explorer has not yet spent its per-level budget:\n                 run frontier_explorer for up to TP4_BUDGET actions (default\n                 800) or until the level changes / game ends. Engine actions\n                 via the gateway cost ~8 ms each, so 800 actions is seconds.\n  ENDGAME        time_remaining <= TP4_ENDGAME_S (default 300) and no level\n                 progress in the last TP4_ENDGAME_QUIET_S (default 900):\n                 run the explorer until TP4_ENDGAME_STOP_S (30) remain.\n\nAfter a run the next user prompt carries a HARNESS NOTE (what was tried, how\nmany actions, whether a level was completed) and Pack 2\'s stall counters are\nreset. Measured offline (bench_explorer.py): alone, the explorer reaches L1 on\n10/25 public games within 800 actions and 15/25 within 3000; on the LLM\'s\nzero-level games it adds 2 (800) to 4 (3000). Its job is unlocking depth for\nthe LLM, not scoring the level itself (an 800-action level scores ~0).\n"""\nfrom __future__ import annotations\n\nimport os\nimport time\nfrom typing import Any\n\n_STATE = {"installed": False}\n_OFF = {"0", "false", "no", "off"}\n_ANALYZE_STOCK: dict[str, Any] = {}\n\nEXPLORER_NOTE = (\n    "HARNESS NOTE: a model-free explorer took over for {n} actions ({why}); it {outcome}. "\n    "It tried {tested} distinct (state, action) pairs across {nodes} board states. "\n    "All of its actions are in `history`. {tail}"\n)\n\n\ndef _env(name: str, default: str) -> str:\n    raw = os.environ.get(name)\n    return default if raw is None or not raw.strip() else raw.strip()\n\n\ndef enabled() -> bool:\n    return _env("TP4_ENABLE", "1").lower() not in _OFF\n\n\ndef _int(name: str, default: int) -> int:\n    try:\n        return int(_env(name, str(default)))\n    except ValueError:\n        return default\n\n\ndef stall_t3() -> int:\n    return max(1, _int("TP4_STALL_T3", 30))\n\n\ndef budget() -> int:\n    return max(1, _int("TP4_BUDGET", 800))\n\n\ndef runs_per_level() -> int:\n    return max(0, _int("TP4_RUNS_PER_LEVEL", 1))\n\n\ndef endgame_s() -> float:\n    return float(_int("TP4_ENDGAME_S", 300))\n\n\ndef endgame_quiet_s() -> float:\n    return float(_int("TP4_ENDGAME_QUIET_S", 900))\n\n\ndef endgame_stop_s() -> float:\n    return float(_int("TP4_ENDGAME_STOP_S", 30))\n\n\ndef status() -> dict[str, Any]:\n    return {"installed": _STATE["installed"], "enabled": enabled(), "stall_t3": stall_t3(),\n            "budget": budget(), "runs_per_level": runs_per_level(), "endgame_s": endgame_s()}\n\n\n# ------------------------------------------------------------- the run ---\ndef run_explorer(session: Any, solver_mod: Any, arcengine: Any, fe: Any, *, max_actions: int,\n                 deadline_s: float | None = None, why: str = "stall") -> dict[str, Any]:\n    """Drive the session\'s engine with the frontier explorer until the level\n    changes, the game ends, the budget is spent, or the deadline passes."""\n    game = session.game\n    level0 = int(solver_mod._level_number(game))\n    ex = fe.FrontierExplorer(seed=int(time.time()) & 0xFFFF)\n    grid = solver_mod._grid_from_state(game.current_state)\n    key = ex.observe(grid, list(game.current_state.available_actions))\n    n = 0\n    outcome = "made no level progress"\n    t0 = time.monotonic()\n    while n < max_actions:\n        if deadline_s is not None and (time.monotonic() - t0) >= deadline_s:\n            outcome = "stopped at the time limit"\n            break\n        try:\n            if session.should_stop():\n                outcome = "stopped (session ending)"\n                break\n        except Exception:  # noqa: BLE001\n            pass\n        if solver_mod._is_engine_game_over(game):\n            action = arcengine.ActionInput(id=arcengine.GameAction.RESET, data={})\n            session._execute_action(action, batch_index=1, batch_size=1, generated_tokens=0)\n            n += 1\n            grid = solver_mod._grid_from_state(game.current_state)\n            key = ex.observe(grid, list(game.current_state.available_actions))\n            continue\n        idx, _why = ex.choose(key)\n        cand = ex.nodes[key].candidates[idx]\n        action = arcengine.ActionInput(id=arcengine.GameAction.from_name(cand.action), data=dict(cand.data))\n        payload = session._execute_action(action, batch_index=1, batch_size=1, generated_tokens=0)\n        n += 1\n        if payload.get("run_complete"):\n            outcome = "COMPLETED THE GAME"\n            break\n        if payload.get("level_completed") or int(solver_mod._level_number(game)) != level0:\n            outcome = f"COMPLETED level {level0}; you are now on level {solver_mod._level_number(game)}"\n            break\n        grid = solver_mod._grid_from_state(game.current_state)\n        new_key = ex.observe(grid, list(game.current_state.available_actions))\n        ex.record(key, idx, new_key)\n        key = new_key\n    tested = ex.stats["edges"] + ex.stats["noops"]\n    return {"actions": n, "outcome": outcome, "tested": tested, "nodes": ex.stats["nodes"],\n            "why": why, "wall_s": round(time.monotonic() - t0, 1)}\n\n\ndef _note(rec: dict[str, Any]) -> str:\n    tail = ("Build on the new level from a fresh look at `current_frame`."\n            if "COMPLETED" in rec["outcome"] else\n            "The explored actions did not progress the level: prefer hypotheses that explain "\n            "why, and try targets or sequences the explorer could not (it never chains actions "\n            "with intent).")\n    return EXPLORER_NOTE.format(n=rec["actions"], why=rec["why"], outcome=rec["outcome"],\n                                tested=rec["tested"], nodes=rec["nodes"], tail=tail)\n\n\n# --------------------------------------------------------------- install ---\ndef install() -> str:\n    if _STATE["installed"]:\n        return "explore: SKIP (already applied)"\n    try:\n        import graft_control as tc  # noqa: PLC0415\n        import frontier_explorer as fe  # noqa: PLC0415\n    except Exception as exc:  # noqa: BLE001\n        return f"explore: SKIP (module missing: {exc!r})"\n    if not tc._STATE.get("installed"):\n        return "explore: SKIP (graft_control not installed)"\n    try:\n        import arcengine  # noqa: PLC0415\n        from inference.agent import tool_agent as agent_mod  # noqa: PLC0415\n        from inference.framework import solver as solver_mod  # noqa: PLC0415\n    except Exception as exc:  # noqa: BLE001\n        return f"explore: SKIP (import failed: {exc!r})"\n    agent_cls = agent_mod.ToolAgent\n    stock_analyze = agent_cls.analyze\n\n    def analyze(self, state_path, action_num, valid_actions=None, step_env=None, **kwargs):\n        try:\n            sess = getattr(step_env, "__self__", None)\n            if sess is not None and enabled():\n                st = tc._state(sess)\n                runs = getattr(st, "explorer_runs_this_level", 0)\n                level = st.level\n                if getattr(st, "explorer_level", None) != level:\n                    st.explorer_level = level\n                    st.explorer_runs_this_level = 0\n                    runs = 0\n                rec = None\n                timing = {}\n                try:\n                    timing = sess.timing_payload()\n                except Exception:  # noqa: BLE001\n                    pass\n                remaining = timing.get("time_remaining_seconds")\n                quiet = time.monotonic() - getattr(st, "last_progress_at", time.monotonic())\n                if (st.since_new >= stall_t3() and getattr(st, "resets_this_level", 0) >= 1\n                        and runs < runs_per_level()):\n                    rec = run_explorer(sess, solver_mod, arcengine, fe, max_actions=budget(),\n                                       deadline_s=None if remaining is None else max(5.0, remaining - endgame_stop_s()),\n                                       why=f"{st.since_new} actions without a new board state")\n                    st.explorer_runs_this_level = runs + 1\n                elif (remaining is not None and remaining <= endgame_s() and quiet >= endgame_quiet_s()\n                      and not getattr(st, "endgame_done", False)):\n                    st.endgame_done = True\n                    rec = run_explorer(sess, solver_mod, arcengine, fe, max_actions=10 ** 6,\n                                       deadline_s=max(1.0, remaining - endgame_stop_s()),\n                                       why="endgame: no level progress recently and the clock is almost out")\n                if rec is not None:\n                    st.since_new = 0\n                    st.streak = 0\n                    st.last_reset_note = _note(rec)\n                    st.explorer_last = rec\n                    try:\n                        sess.write_runtime_state()\n                    except Exception:  # noqa: BLE001\n                        pass\n                    action_num = sess.action_count\n                    valid_actions = solver_mod._engine_action_names(sess.game)\n        except Exception:  # noqa: BLE001\n            pass\n        return _ANALYZE_STOCK["fn"](self, state_path, action_num, valid_actions=valid_actions,\n                                    step_env=step_env, **kwargs)\n\n    _ANALYZE_STOCK["fn"] = stock_analyze\n    analyze._tp4_stock = stock_analyze\n    agent_cls.analyze = analyze\n\n    # progress clock for the endgame trigger: any level change stamps it\n    stock_after = tc._after_action\n\n    def after_action(st, before, after, payload):\n        prev_level = st.level\n        stock_after(st, before, after, payload)\n        if st.level != prev_level or not hasattr(st, "last_progress_at"):\n            st.last_progress_at = time.monotonic()\n\n    after_action._tp4_stock = stock_after\n    tc._after_action = after_action\n\n    _STATE["installed"] = True\n    return "explore: OK"\n', 'graft_emission.py': '"""Emission graft (Pack 5, 2026-08-30) — attack the two measured modal\nfailures of the stock loop (docs/research-2026-08-29/R7-stock-failure-\nforensics-2026-08-30.md):\n\n  G  analysis-paralysis: 63% of wall time sits in model calls that execute\n     ZERO env actions; 47% of calls end at the 60 s yield without acting\n     (tn36: 50 identical calls, 0 actions in 2.2 h).\n  amnesia-by-channel: in 4/16 games the assistant text is empty all run, so\n     the note harvest gets nothing and the model restarts from scratch every\n     call (re86: 54 responses with 0 content chars). The world model lives in\n     the hidden REASONING channel (Feng\'s 66.8% field finding).\n\nTwo independent, flag-gated behaviours (installed after graft_throughput;\nTP5_ENABLE=0 = pass-through):\n\n  (A) TP5_WM_FROM_REASONING (default 1)\n      When a model response carries NO parsable note in its assistant text,\n      harvest `World model:`-style labelled blocks from the reasoning text\n      instead. Purely additive: the assistant channel wins when non-empty.\n\n  (B) TP5_ACT_FLOOR (default 3)\n      Track consecutive model calls WITHOUT an executed env action, across\n      turns, per agent (the session\'s own counter — reset whenever an action\n      executes). When the streak reaches the floor, the NEXT request forces\n      `tool_choice` to the python function and appends one user line telling\n      the model to act on its best current hypothesis. Mechanism reused from\n      submission/_effort_medium/graft_effort.py (the dead-retry seam), which\n      passed its offline suite; the trigger here is broader (no action\n      executed, not just empty completions) and the injected line names the\n      analysis streak. TP5_ACT_FLOOR=0 disables.\n\nFail-open: errors fall through to stock behaviour.\n"""\nfrom __future__ import annotations\n\nimport os\nimport threading\nfrom typing import Any\n\n_STATE = {"installed": False}\n_RUN_STOCK: dict = {}\n_OFF = {"0", "false", "no", "off"}\n_tls = threading.local()\n\nACT_LINE = (\n    "You have made {n} analyses in a row without executing any action. Analysis is no longer "\n    "buying information the board can\'t give you faster. Call the `python` tool NOW and make it "\n    "end with `action(...)` executing the best action or short batch under your current best "\n    "hypothesis — acting and observing the result IS the experiment."\n)\nFORCED_TOOL_CHOICE = {"type": "function", "function": {"name": "python"}}\n\n\ndef _env(name: str, default: str) -> str:\n    raw = os.environ.get(name)\n    return default if raw is None or not raw.strip() else raw.strip()\n\n\ndef enabled() -> bool:\n    return _env("TP5_ENABLE", "1").lower() not in _OFF\n\n\ndef wm_from_reasoning() -> bool:\n    return enabled() and _env("TP5_WM_FROM_REASONING", "1").lower() not in _OFF\n\n\ndef act_floor() -> int:\n    if not enabled():\n        return 0\n    try:\n        return max(0, int(_env("TP5_ACT_FLOOR", "3")))\n    except ValueError:\n        return 3\n\n\ndef status() -> dict[str, Any]:\n    return {"installed": _STATE["installed"], "enabled": enabled(),\n            "wm_from_reasoning": wm_from_reasoning(), "act_floor": act_floor()}\n\n\ndef _calls_without_action(agent: Any) -> int:\n    return int(getattr(agent, "_tp5_calls_without_action", 0) or 0)\n\n\ndef install() -> str:\n    if _STATE["installed"]:\n        return "emission: SKIP (already applied)"\n    try:\n        from inference.agent import tool_agent as agent_mod\n        from inference.utils import openai_compat as compat_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"emission: SKIP (import failed: {exc!r})"\n    agent_cls = getattr(agent_mod, "ToolAgent", None)\n    if agent_cls is None:\n        return "emission: SKIP (missing ToolAgent)"\n    for name in ("_update_summarized_knowledge_from_assistant", "_chat_completion",\n                 "_run_python_tool", "_extract_scientist_note"):\n        if getattr(agent_cls, name, getattr(agent_mod, name, None)) is None:\n            return f"emission: SKIP ({name} missing)"\n    if getattr(agent_mod, "build_chat_payload", None) is None:\n        return "emission: SKIP (build_chat_payload rebind seam missing)"\n\n    # (A) note harvest falls back to the reasoning channel ------------------\n    stock_chat = agent_cls._chat_completion\n\n    def chat(self, messages, **kwargs):\n        # act-floor: arm the forced tool choice for THIS request when the\n        # analysis streak has reached the floor\n        floor = act_floor()\n        force = floor > 0 and _calls_without_action(self) >= floor\n        _tls.force_now = force\n        _tls.act_line_n = _calls_without_action(self)\n        try:\n            if force:\n                messages = list(messages) + [{"role": "user", "content": ACT_LINE.format(n=_tls.act_line_n)}]\n            result = stock_chat(self, messages, **kwargs)\n        finally:\n            _tls.force_now = False\n        try:\n            self._tp5_calls_without_action = _calls_without_action(self) + 1\n            if wm_from_reasoning():\n                message = getattr(result, "message", None)\n                if isinstance(message, dict):\n                    content = agent_mod._normalize_message_content(message.get("content", ""))\n                    if not agent_mod._extract_scientist_note(str(content or "")):\n                        reasoning = str(agent_mod._extract_reasoning_text(message) or "")\n                        # J10 re-verify (iv): a Next:/Suggestion: intent line in\n                        # the reasoning channel must not glue into stock fields —\n                        # route through TP10\'s strip when that graft is present.\n                        try:\n                            import graft_memoryspine as _tp10\n                            reasoning, _intent = _tp10.split_intent(reasoning)\n                        except Exception:  # noqa: BLE001\n                            pass\n                        note = agent_mod._extract_scientist_note(reasoning)\n                        if note:\n                            for key, value in note.items():\n                                if value:\n                                    self._summarized_knowledge[key] = value\n        except Exception:  # noqa: BLE001\n            pass\n        return result\n\n    chat._tp5_stock = stock_chat\n    agent_cls._chat_completion = chat\n\n    # forced tool choice rides build_chat_payload only while armed ----------\n    stock_build = compat_mod.build_chat_payload\n\n    def build(*args, **kwargs):\n        try:\n            if getattr(_tls, "force_now", False) and kwargs.get("tools"):\n                kwargs = dict(kwargs)\n                kwargs["tool_choice"] = FORCED_TOOL_CHOICE\n        except Exception:  # noqa: BLE001\n            pass\n        return stock_build(*args, **kwargs)\n\n    build._tp5_stock = stock_build\n    compat_mod.build_chat_payload = build\n    agent_mod.build_chat_payload = build  # dual-namespace rebind (imported by name)\n\n    # streak reset: an executed action clears the counter -------------------\n    _RUN_STOCK["fn"] = agent_cls._run_python_tool\n\n    def run_tool(self, state_path, arguments):\n        result = _RUN_STOCK["fn"](self, state_path, arguments)\n        try:\n            if getattr(result, "step_executed", False):\n                self._tp5_calls_without_action = 0\n        except Exception:  # noqa: BLE001\n            pass\n        return result\n\n    run_tool._tp5_stock = _RUN_STOCK["fn"]\n    agent_cls._run_python_tool = run_tool\n\n    _STATE["installed"] = True\n    return "emission: OK"\n', 'graft_economy.py': '"""Action-economy graft (TP6, 2026-08-31) — tell the model the truth about\nscoring, and surface its own action spend.\n\nMeasured leak (submission/_flashnext_smoke/results_v4 + scoring math): on the\nFlash-Next smoke, 1.63 local pts/game are forfeited on COMPLETED levels done\nfar over the human baseline (ka59 L1 296 acts vs 28 -> 0.03 pts; bp35 189/21;\nsc25 83/36; quadratic penalty). The stock duck prompt NEVER mentions that\nactions are scored, so the model optimises for progress only.\n\nOne flag-gated behaviour (TP6_ENABLE, default 1): append a short ACTION\nECONOMY block to the user prompt each turn:\n  - actions are scored: per level, score = (human_baseline / your_actions)^2,\n    so finishing a level in 2x the needed actions costs 75% of its points;\n  - the current level\'s action count so far (from actions_per_level via the\n    step summary when available, else the turn-header action counter);\n  - three rules: probe ONCE per hypothesis, never repeat an action that\n    already changed nothing (results carry board_changed), and once the\n    mechanic is verified execute the remaining plan as ONE batch.\n\nPurely prompt-side; no seam behaviour changes. Installed on ToolAgent.\n_build_user_prompt after any other graft (append-only, fail-open).\n"""\nfrom __future__ import annotations\n\nimport os\nfrom typing import Any\n\n_STATE = {"installed": False}\n_OFF = {"0", "false", "no", "off"}\n\nECONOMY_BLOCK = (\n    "ACTION ECONOMY (scoring truth): every level is scored (human_baseline / your_actions)^2 — "\n    "taking twice the needed actions keeps only a quarter of the level\'s points, and points are "\n    "only awarded for COMPLETED levels. You have executed {n} actions on this level so far. "\n    "Probe once per hypothesis; never repeat an action that already changed nothing; once a "\n    "mechanic is verified, execute the whole remaining plan as one action([...]) batch."\n)\n\n\ndef enabled() -> bool:\n    raw = os.environ.get("TP6_ENABLE")\n    value = "1" if raw is None or not raw.strip() else raw.strip()\n    return value.lower() not in _OFF\n\n\ndef status() -> dict[str, Any]:\n    return {"installed": _STATE["installed"], "enabled": enabled()}\n\n\ndef _level_actions(agent: Any, action_num: int) -> int:\n    try:\n        summary = getattr(agent, "_last_step_summary", None) or {}\n        end = summary.get("end_action_num")\n        # best available proxy: total actions this game; per-level split is not\n        # visible to the agent, so state the game counter when level unknown\n        return int(end if end is not None else max(0, action_num))\n    except Exception:  # noqa: BLE001\n        return max(0, int(action_num or 0))\n\n\ndef install() -> str:\n    if _STATE["installed"]:\n        return "economy: SKIP (already applied)"\n    try:\n        from inference.agent import tool_agent as agent_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"economy: SKIP (import failed: {exc!r})"\n    cls = getattr(agent_mod, "ToolAgent", None)\n    if cls is None or getattr(cls, "_build_user_prompt", None) is None:\n        return "economy: SKIP (seam missing)"\n\n    stock = cls._build_user_prompt\n\n    def build(self, action_num, **kwargs):\n        text = stock(self, action_num, **kwargs)\n        if not enabled():\n            return text\n        try:\n            return text + "\\n" + ECONOMY_BLOCK.format(n=_level_actions(self, action_num))\n        except Exception:  # noqa: BLE001\n            return text\n\n    build._tp6_stock = stock\n    cls._build_user_prompt = build\n    _STATE["installed"] = True\n    return "economy: OK"\n', 'graft_deaths.py': '"""Death-protocol graft (TP7, 2026-08-31) — plan around per-level move budgets.\n\nMeasured (docs/research-2026-08-29/R8-flashnext-forensics-2026-08-31.md): on\nthe Flash-Next smoke, 72.5% of ALL actions sat inside attempts that ended in a\nGAME_OVER; the deaths come at FIXED per-level cadences (sc25 every 27 actions,\nvc33 every 51, ka59 ~100, wa30 ~200 — move budgets, not hazards). The model\nnamed the cadence in 2 of 8 games and planned against it in 0; retries replay\nthe same ground from scratch (retry actions ACCUMULATE into the level\'s scored\naction count).\n\nOne flag-gated behaviour (TP7_ENABLE, default 1): track GAME_OVERs per level\non the session (actions at each death; gaps between deaths). After the first\ndeath on a level, append a DEATH PROTOCOL block to the user prompt:\n  - name the event: likely a per-level MOVE BUDGET or timer, not (only) a\n    hazard; retried actions still count against the level\'s score;\n  - after >=2 deaths with a stable gap, state the estimated budget N and\n    actions already spent this life;\n  - the rule: probe on an early life, BANK a complete plan in your notes,\n    then execute it within ONE life as one batch.\n\nSession state rides the graft_control SessionState when present, else its own\nattribute. Prompt seam: ToolAgent._build_user_prompt (append-only, fail-open);\ndeath detection: _HarnessGameSession._execute_action wrapper.\n"""\nfrom __future__ import annotations\n\nimport os\nfrom typing import Any\n\n_STATE = {"installed": False}\n_OFF = {"0", "false", "no", "off"}\n\nPROTOCOL_FIRST = (\n    "DEATH PROTOCOL: this level has had {k} GAME_OVER(s) (at level-action counts {at}). "\n    "On these games a GAME_OVER is usually a per-level MOVE BUDGET or timer expiring — not "\n    "necessarily a mistake you made. Retried actions STILL COUNT against this level\'s score. "\n    "Treat lives as experiment slots: probe on an early life, bank a COMPLETE plan in your notes, "\n    "then execute it within one life as a single batch."\n)\nPROTOCOL_BUDGET = (\n    " Estimated budget: ~{budget} actions per life (stable death cadence); you have spent "\n    "{spent} actions this life — keep the executing plan within the remaining ~{left}."\n)\n\n\ndef enabled() -> bool:\n    raw = os.environ.get("TP7_ENABLE")\n    value = "1" if raw is None or not raw.strip() else raw.strip()\n    return value.lower() not in _OFF\n\n\ndef status() -> dict[str, Any]:\n    return {"installed": _STATE["installed"], "enabled": enabled()}\n\n\nclass DeathState:\n    def __init__(self) -> None:\n        self.level: int | None = None\n        self.level_actions = 0          # actions on the current level (all lives)\n        self.death_at: list[int] = []   # level_actions at each GAME_OVER\n\n\ndef _dstate(session: Any) -> DeathState:\n    st = getattr(session, "_tp7", None)\n    if st is None:\n        st = DeathState()\n        try:\n            session._tp7 = st\n        except Exception:  # noqa: BLE001\n            pass\n    return st\n\n\ndef _session_of(agent: Any) -> Any:\n    cb = getattr(agent, "_step_env_callback", None)\n    return getattr(cb, "__self__", None)\n\n\ndef estimate_budget(death_at: list[int]) -> int | None:\n    """Stable cadence: gaps between consecutive deaths within 20% of each other."""\n    if len(death_at) < 2:\n        return None\n    gaps = [b - a for a, b in zip(death_at, death_at[1:])]\n    gaps = [g for g in gaps if g > 0]\n    if not gaps:\n        return None\n    mean = sum(gaps) / len(gaps)\n    if all(abs(g - mean) <= 0.2 * mean + 2 for g in gaps):\n        return round(mean)\n    # fall back to the first death position if gaps are noisy but positive\n    return None\n\n\ndef protocol_text(st: DeathState) -> str | None:\n    if not st.death_at:\n        return None\n    text = PROTOCOL_FIRST.format(k=len(st.death_at), at=st.death_at[-4:])\n    budget = estimate_budget(st.death_at) or (st.death_at[0] if len(st.death_at) == 1 else None)\n    if budget:\n        spent = max(0, st.level_actions - st.death_at[-1])\n        text += PROTOCOL_BUDGET.format(budget=budget, spent=spent, left=max(0, budget - spent))\n    return text\n\n\ndef install() -> str:\n    if _STATE["installed"]:\n        return "deaths: SKIP (already applied)"\n    try:\n        from inference.agent import tool_agent as agent_mod\n        from inference.framework import solver as solver_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"deaths: SKIP (import failed: {exc!r})"\n    agent_cls = agent_mod.ToolAgent\n    session_cls = solver_mod._HarnessGameSession\n    if getattr(agent_cls, "_build_user_prompt", None) is None or getattr(session_cls, "_execute_action", None) is None:\n        return "deaths: SKIP (seam missing)"\n\n    stock_execute = session_cls._execute_action\n\n    def execute(self, action, *args, **kwargs):\n        payload = stock_execute(self, action, *args, **kwargs)\n        if not enabled():\n            return payload\n        try:\n            st = _dstate(self)\n            level = payload.get("level")\n            if level is not None and level != st.level:\n                st.level = level\n                st.level_actions = 0\n                st.death_at = []\n            if payload.get("executed"):\n                st.level_actions += 1\n            if payload.get("game_over"):\n                st.death_at.append(st.level_actions)\n        except Exception:  # noqa: BLE001\n            pass\n        return payload\n\n    execute._tp7_stock = stock_execute\n    session_cls._execute_action = execute\n\n    stock_prompt = agent_cls._build_user_prompt\n\n    def build(self, action_num, **kwargs):\n        text = stock_prompt(self, action_num, **kwargs)\n        if not enabled():\n            return text\n        try:\n            sess = _session_of(self)\n            if sess is not None:\n                block = protocol_text(_dstate(sess))\n                if block:\n                    text = text + "\\n" + block\n        except Exception:  # noqa: BLE001\n            pass\n        return text\n\n    build._tp7_stock = stock_prompt\n    agent_cls._build_user_prompt = build\n\n    _STATE["installed"] = True\n    return "deaths: OK"\n', 'graft_durable.py': '"""Durable-timeout graft (TP8, 2026-08-31) — a timed-out python tool call\nmust not discard what its actions observed.\n\nMeasured (R8 forensics): ~130 tool timeouts across 11 Flash-Next games\n(wa30 46/117 turns, ar25 22 of its last 32). The sandbox returns the executed\nactions\' payloads even on timeout (python_tool_sandbox.run_sandboxed_python\ntimeout paths carry host_action_results), but ToolAgent._run_python_tool\'s\nerror branch shows the model ONLY "Tool timed out after 30s" — the outcomes\n(board_changed, level, state) are dropped, and the model dead-reckons\n(ar25\'s terminal net-zero oscillation; ft09\'s last 9 actions swallowed).\n\nFix, one seam: rebind run_sandboxed_python inside tool_agent; when the result\nis a timeout WITH executed actions, rewrite the error to a recap naming the\nexecuted count and the last action\'s outcome, so the stock error branch\nrenders it to the model. TP8_ENABLE=0 = pass-through.\n"""\nfrom __future__ import annotations\n\nimport os\nfrom typing import Any\n\n_STATE = {"installed": False}\n_STOCK: dict = {}\n_OFF = {"0", "false", "no", "off"}\n\nRECAP = (\n    "{error} — HOWEVER {n} action(s) DID execute before the timeout and their effects are real. "\n    "Last executed action result: level={level}, state={state}, board_changed={bc}, score={score}, "\n    "level_completed={lc}, game_over={go}. `current_frame` already reflects these actions — "\n    "re-observe it instead of re-running the actions."\n)\n\n\ndef enabled() -> bool:\n    raw = os.environ.get("TP8_ENABLE")\n    value = "1" if raw is None or not raw.strip() else raw.strip()\n    return value.lower() not in _OFF\n\n\ndef status() -> dict[str, Any]:\n    return {"installed": _STATE["installed"], "enabled": enabled()}\n\n\ndef _recap(result: dict[str, Any]) -> dict[str, Any]:\n    try:\n        error = str(result.get("error") or "")\n        if "timed out" not in error.lower():\n            return result\n        executed = [r for r in (result.get("action_results") or [])\n                    if isinstance(r, dict) and r.get("executed")]\n        if not executed:\n            return result\n        last = executed[-1]\n        out = dict(result)\n        out["error"] = RECAP.format(\n            error=error.rstrip("."), n=len(executed),\n            level=last.get("level"), state=last.get("state"), bc=last.get("board_changed"),\n            score=last.get("score"), lc=last.get("level_completed"), go=last.get("game_over"),\n        )\n        return out\n    except Exception:  # noqa: BLE001\n        return result\n\n\ndef install() -> str:\n    if _STATE["installed"]:\n        return "durable: SKIP (already applied)"\n    try:\n        from inference.agent import tool_agent as agent_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"durable: SKIP (import failed: {exc!r})"\n    stock = getattr(agent_mod, "run_sandboxed_python", None)\n    if stock is None:\n        return "durable: SKIP (run_sandboxed_python not bound in tool_agent)"\n    _STOCK["fn"] = stock\n\n    def run(*args, **kwargs):\n        result = _STOCK["fn"](*args, **kwargs)\n        if not enabled():\n            return result\n        if isinstance(result, dict):\n            return _recap(result)\n        return result\n\n    run._tp8_stock = stock\n    agent_mod.run_sandboxed_python = run\n    _STATE["installed"] = True\n    return "durable: OK"\n', 'graft_pipeline.py': '"""Turn-pipeline repair graft (TP9, 2026-08-31) — no discarded thinking,\nlivelock breaker, one client-timeout retry.\n\nEvidence (docs/research-2026-08-31/R9-path-to-7-gap-analysis.md §2.1): 48% of\nall LLM turns are idle (`step_executed: False`), overwhelmingly "Yielded\ncontrol to solver: turn_time_budget"; 13 vLLM read-timeout turns stranded\ngames at run end; r11l sat in a 6,000 s deterministic livelock (20+\nconsecutive idle turns with byte-identical transcript lengths) after clearing\nL1 in 5 actions.\n\nSeams, verified in the anim bundle (the code that plays at eval —\nsubmission/_inspect_replay/assets_build/ARC3-Inference):\n\n- THE YIELD. tool_agent.py:154 reads LOCAL_ANALYZER_YIELD_SECONDS (60 in the\n  measured run) into :1085 `self._yield_seconds`; `control_yield_reason`\n  (:2154-2163) returns "turn_time_budget" once the turn is past it. The check\n  fires between requests (:2168 loop top, :2288 after a no-tool-call reply,\n  :2353 between tool calls) and breaks out of `analyze`.\n\n- WHERE THE THINKING DIES. On a `requests.RequestException` (read-timeout\n  included) `analyze` sets preserve_history=False (:2365), reverts\n  `_history_messages` to the pre-turn snapshot (:2403), and returns\n  `AnalyzerTurnResult(step_executed=False, retryable_failure=True,\n  reasoning=captured_reasoning)` (:2380) — the ONLY surviving copy of the\n  turn\'s reasoning is `result.reasoning`. The solver loop (solver.py:352-363)\n  handles retryable_failure / yielded_control / idle by `continue` and NEVER\n  reads `result.reasoning`: that field is dropped on the floor every idle\n  turn. (On a clean turn_time_budget yield after a no-tool-call reply the\n  reasoning-only assistant message does land in `_history_messages`, but the\n  next `analyze` rebuilds context from a trimmed history and — at fixed\n  seed/temperature — regenerates the same doomed turn; r11l\'s identical\n  transcript lengths are exactly that. So even the "kept" case needs the\n  explicit resume-and-act nudge, and the kept copy can be trimmed away.)\n\n- THE HTTP CALL. `_chat_completion` (:1518-1546) does one `requests.post`\n  with timeout=`request_timeout_seconds` (solver.py:267-284: min of analyzer\n  timeout, wall remaining, soft remaining). A ReadTimeout surfaces as\n  `requests.RequestException` -> the discard path above; the solver retries\n  the analysis step after a 1 s backoff (solver.py:64,352-357) but the\n  generation is already lost, and at run end `should_stop()` breaks first\n  (solver.py:354-355) — the 13 stranded turns.\n\nThree behaviours, each individually gated (TP9_ENABLE=0 turns all off):\n\n(a) TURN PERSIST/RESUME (TP9_RESUME, default 1). Wrap `ToolAgent.analyze`:\n    when the stock turn comes back idle (yielded_control or\n    retryable_failure) with non-empty `result.reasoning`, stash the tail\n    (TP9_RESUME_CHARS, default 1500) on the agent. Wrap\n    `ToolAgent._build_user_prompt`: the next turn\'s user prompt gets a\n    RESUME block carrying that tail plus an instruction to continue from it\n    and act — consumed once, cleared on any executed step, reset on game\n    change (mirrors _ensure_session\'s runtime-dir keying, :1140-1152).\n    J9/F6: a consumed tail is parked, not dropped — if the very request it\n    was injected into dies with NO new reasoning (timeout before any reply),\n    the parked tail is restored and re-injected next turn; any new reasoning\n    or an executed step discards the parked copy.\n\n(b) LIVELOCK DETECTOR (TP9_LIVELOCK, default 1; TP9_LIVELOCK_K, default 3;\n    TP9_LIVELOCK_COOLDOWN, default 5). Per game, hash each idle turn\'s\n    `result.reasoning` (empty output hashes equal — that IS the r11l\n    signature). K consecutive identical hashes with zero executed actions\n    arms a LOOP BREAKER block for the next user prompt ("state ONE new\n    hypothesis and emit one action([...]) batch now") and bumps a counter.\n    Streak resets on an executed step or a differing hash.\n    J9/F3a: `retryable_failure` turns are EXCLUDED from the streak — they\n    carry no model-produced output (a 3 s server outage at the solver\'s 1 s\n    retry cadence must never arm a false "identical output" accusation);\n    they neither build nor reset the streak.\n    J9/F3b: the breaker fires AT MOST ONCE per armed streak — injection\n    resets the streak and starts a cooldown of TP9_LIVELOCK_COOLDOWN counted\n    turns during which no streak accumulates, bounding the injection rate to\n    1 per (K + cooldown) turns (~13 per 100-turn persistent livelock at\n    defaults, vs every prompt before the fix).\n    Scope (J9/F5): hash identity only catches the empty/deterministic-replay\n    subspecies; a livelock that still emits sampled non-identical text at\n    temp 0.6 is out of scope for this detector.\n\n(c) CLIENT-TIMEOUT RETRY (TP9_RETRY, default 1; TP9_RETRY_BACKOFF, default\n    1.0 s; TP9_RETRY_MIN_BUDGET, default 5.0 s). Wrap `_chat_completion`:\n    retry `requests.Timeout` / `requests.ConnectionError` up to TP9_RETRY\n    times before letting the exception reach the discard path. Other\n    RequestExceptions (HTTP errors, context-length rejections handled at\n    :2221) pass through untouched — a context-overflow retry of the\n    identical payload deterministically fails and the analyze loop has its\n    own recovery for it.\n    J9/F4 wall-clock guard: the turn\'s `request_timeout_seconds` is the\n    solver\'s min(analyzer timeout, wall remaining, soft remaining)\n    (solver.py:267-284), so it IS the wall bound. The retry never exceeds\n    it: the retry attempt gets only the leftover budget\n    (budget - elapsed - backoff), and when that leftover is under\n    TP9_RETRY_MIN_BUDGET the retry is skipped and the exception propagates\n    (stock behaviour, letting the solver\'s own should_stop-guarded retry\n    path take over). Net effect: a fast ConnectionError still gets its\n    near-free retry; a ReadTimeout that consumed the whole budget is never\n    doubled. With no budget argument and no analyzer timeout the retry is\n    unbounded, as before.\n\nConventions: module-level _STOCK dict (never only a closure), fail-open\ntry/except around all graft logic, rebinding class attributes only, new file\nonly. install() -> "pipeline: OK" / "pipeline: SKIP (...)".\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport os\nimport time\nfrom typing import Any\n\n_STATE: dict[str, Any] = {\n    "installed": False,\n    "resumes_injected": 0,\n    "perturbations_injected": 0,\n    "retries_used": 0,\n}\n_STOCK: dict[str, Any] = {}\n_OFF = {"0", "false", "no", "off"}\n\nDEFAULT_LIVELOCK_K = 3\nDEFAULT_LIVELOCK_COOLDOWN = 5\nDEFAULT_RETRY = 1\nDEFAULT_RESUME_CHARS = 1500\nDEFAULT_RETRY_BACKOFF = 1.0\nDEFAULT_RETRY_MIN_BUDGET = 5.0\n\nRESUME_BLOCK = (\n    "RESUME: You were interrupted mid-thought last turn ({reason}) and nothing was applied. "\n    "Your partial reasoning from that turn (tail):\\n<<<\\n{tail}\\n>>>\\n"\n    "Do not re-derive this from scratch. Continue from those conclusions and emit a `python` "\n    "tool call that reaches `action(actions)` promptly this turn."\n)\nPERTURB_BLOCK = (\n    "LOOP BREAKER: You have produced identical output {n} times in a row with no actions "\n    "executed. Break the loop: state ONE new hypothesis in a single sentence, then emit one "\n    "`python` tool call that calls `action([...])` with a small batch NOW — act first, "\n    "analyze the result after."\n)\n\n\n# ----------------------------------------------------------------- flags ---\ndef _env(name: str, default: str) -> str:\n    raw = os.environ.get(name)\n    return default if raw is None or not raw.strip() else raw.strip()\n\n\ndef enabled() -> bool:\n    return _env("TP9_ENABLE", "1").lower() not in _OFF\n\n\ndef resume_enabled() -> bool:\n    return enabled() and _env("TP9_RESUME", "1").lower() not in _OFF\n\n\ndef livelock_enabled() -> bool:\n    return enabled() and _env("TP9_LIVELOCK", "1").lower() not in _OFF\n\n\ndef livelock_k() -> int:\n    try:\n        return max(2, int(_env("TP9_LIVELOCK_K", str(DEFAULT_LIVELOCK_K))))\n    except ValueError:\n        return DEFAULT_LIVELOCK_K\n\n\ndef livelock_cooldown() -> int:\n    try:\n        return max(0, int(_env("TP9_LIVELOCK_COOLDOWN", str(DEFAULT_LIVELOCK_COOLDOWN))))\n    except ValueError:\n        return DEFAULT_LIVELOCK_COOLDOWN\n\n\ndef retry_attempts() -> int:\n    if not enabled():\n        return 0\n    try:\n        return max(0, int(_env("TP9_RETRY", str(DEFAULT_RETRY))))\n    except ValueError:\n        return DEFAULT_RETRY\n\n\ndef retry_backoff() -> float:\n    try:\n        return max(0.0, float(_env("TP9_RETRY_BACKOFF", str(DEFAULT_RETRY_BACKOFF))))\n    except ValueError:\n        return DEFAULT_RETRY_BACKOFF\n\n\ndef retry_min_budget() -> float:\n    try:\n        return max(0.0, float(_env("TP9_RETRY_MIN_BUDGET", str(DEFAULT_RETRY_MIN_BUDGET))))\n    except ValueError:\n        return DEFAULT_RETRY_MIN_BUDGET\n\n\ndef resume_chars() -> int:\n    try:\n        return max(200, int(_env("TP9_RESUME_CHARS", str(DEFAULT_RESUME_CHARS))))\n    except ValueError:\n        return DEFAULT_RESUME_CHARS\n\n\ndef status() -> dict[str, Any]:\n    return {\n        "installed": _STATE["installed"],\n        "enabled": enabled(),\n        "resume": resume_enabled(),\n        "livelock": livelock_enabled(),\n        "livelock_k": livelock_k(),\n        "livelock_cooldown": livelock_cooldown(),\n        "retry": retry_attempts(),\n        "retry_min_budget": retry_min_budget(),\n        "resume_chars": resume_chars(),\n        "resumes_injected": _STATE["resumes_injected"],\n        "perturbations_injected": _STATE["perturbations_injected"],\n        "retries_used": _STATE["retries_used"],\n    }\n\n\n# ----------------------------------------------------------- per-game state ---\nclass PipelineState:\n    def __init__(self) -> None:\n        self.runtime_dir: Any = None\n        self.resume_tail: str | None = None\n        self.resume_reason: str = ""\n        # F6: the last-injected tail, parked so it can be restored if the\n        # request it rode on dies with no new reasoning.\n        self.injected_tail: str | None = None\n        self.injected_reason: str = ""\n        self.last_hash: str | None = None\n        self.idle_streak = 0\n        self.perturb_pending = False\n        # F3b: counted turns left before the livelock streak may build again.\n        self.cooldown = 0\n\n\ndef _pstate(agent: Any, state_path: Any = None) -> PipelineState:\n    """Get (or create) the agent\'s TP9 state; reset it when the game changes.\n\n    Mirrors ToolAgent._ensure_session, which keys a session by\n    state_path.parent (the per-game runtime dir).\n    """\n    st = getattr(agent, "_tp9", None)\n    if st is None:\n        st = PipelineState()\n        try:\n            agent._tp9 = st\n        except Exception:  # noqa: BLE001\n            pass\n    if state_path is not None:\n        try:\n            runtime_dir = state_path.parent\n            if st.runtime_dir is not None and st.runtime_dir != runtime_dir:\n                fresh = PipelineState()\n                fresh.runtime_dir = runtime_dir\n                try:\n                    agent._tp9 = fresh\n                except Exception:  # noqa: BLE001\n                    pass\n                return fresh\n            st.runtime_dir = runtime_dir\n        except Exception:  # noqa: BLE001\n            pass\n    return st\n\n\ndef _idle_reason(result: Any) -> str:\n    if getattr(result, "retryable_failure", False):\n        return "the model-server request failed"\n    if getattr(result, "yielded_control", False):\n        return "the turn time budget expired"\n    return "no action call was captured"\n\n\ndef _note_turn_result(st: PipelineState, result: Any) -> None:\n    """Update resume/livelock state from one stock analyze() result."""\n    if getattr(result, "step_executed", False):\n        st.resume_tail = None\n        st.resume_reason = ""\n        st.injected_tail = None\n        st.injected_reason = ""\n        st.last_hash = None\n        st.idle_streak = 0\n        st.perturb_pending = False\n        st.cooldown = 0\n        return\n    reasoning = str(getattr(result, "reasoning", "") or "")\n    if resume_enabled():\n        if reasoning.strip():\n            st.resume_tail = reasoning[-resume_chars():]\n            st.resume_reason = _idle_reason(result)\n            st.injected_tail = None\n            st.injected_reason = ""\n        elif st.injected_tail and getattr(result, "retryable_failure", False):\n            # F6 (re-judged): restore the parked tail ONLY when the request it\n            # rode on actually died (retryable_failure). A clean empty yield\n            # means the model saw the nudge and ignored it — re-serving there\n            # stacks duplicate RESUME blocks in a livelock (J9 regression iii).\n            st.resume_tail = st.injected_tail\n            st.resume_reason = st.injected_reason\n    if livelock_enabled():\n        if getattr(result, "retryable_failure", False):\n            # F3a: a failed request produced no model output — a server\n            # outage must neither build nor reset the identical-output streak.\n            return\n        if st.cooldown > 0:\n            # F3b: cooling down after an injection; no streak accumulation.\n            st.cooldown -= 1\n            st.last_hash = None\n            st.idle_streak = 0\n            return\n        digest = hashlib.sha1(reasoning.encode("utf-8", errors="replace")).hexdigest()\n        if digest == st.last_hash:\n            st.idle_streak += 1\n        else:\n            st.idle_streak = 1\n            st.last_hash = digest\n        if st.idle_streak >= livelock_k():\n            st.perturb_pending = True\n\n\n# --------------------------------------------------------------- install ---\ndef install() -> str:\n    if _STATE["installed"]:\n        return "pipeline: SKIP (already applied)"\n    try:\n        from inference.agent import tool_agent as agent_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"pipeline: SKIP (import failed: {exc!r})"\n    try:\n        import requests\n    except Exception as exc:  # noqa: BLE001\n        return f"pipeline: SKIP (requests missing: {exc!r})"\n    cls = getattr(agent_mod, "ToolAgent", None)\n    if cls is None:\n        return "pipeline: SKIP (missing ToolAgent)"\n    for name in ("analyze", "_build_user_prompt", "_chat_completion"):\n        if getattr(cls, name, None) is None:\n            return f"pipeline: SKIP (ToolAgent.{name} missing)"\n\n    _STOCK["analyze"] = cls.analyze\n    _STOCK["build_user_prompt"] = cls._build_user_prompt\n    _STOCK["chat_completion"] = cls._chat_completion\n\n    # -- (a)+(b) capture: wrap analyze -------------------------------------\n    def analyze(self, state_path, action_num, *args, **kwargs):\n        if enabled():\n            try:\n                # Reset per-game state BEFORE the turn so a stale resume tail\n                # from the previous game never leaks into this game\'s prompt.\n                _pstate(self, state_path)\n            except Exception:  # noqa: BLE001\n                pass\n        result = _STOCK["analyze"](self, state_path, action_num, *args, **kwargs)\n        if not enabled() or result is None:\n            return result\n        try:\n            _note_turn_result(_pstate(self, state_path), result)\n        except Exception:  # noqa: BLE001\n            pass\n        return result\n\n    # -- (a)+(b) injection: wrap _build_user_prompt ------------------------\n    def build_user_prompt(self, action_num, **kwargs):\n        text = _STOCK["build_user_prompt"](self, action_num, **kwargs)\n        if not enabled():\n            return text\n        try:\n            st = getattr(self, "_tp9", None)\n            if st is None:\n                return text\n            blocks: list[str] = []\n            if resume_enabled() and st.resume_tail:\n                blocks.append(RESUME_BLOCK.format(reason=st.resume_reason, tail=st.resume_tail))\n                # consumed once — but parked (F6) so a dead injected request\n                # can restore it; discarded on new reasoning or executed step.\n                st.injected_tail = st.resume_tail\n                st.injected_reason = st.resume_reason\n                st.resume_tail = None\n                st.resume_reason = ""\n                _STATE["resumes_injected"] += 1\n            if livelock_enabled() and st.perturb_pending:\n                blocks.append(PERTURB_BLOCK.format(n=st.idle_streak))\n                st.perturb_pending = False\n                # F3b: at most one injection per armed streak — reset the\n                # streak and start the re-arm cooldown.\n                st.idle_streak = 0\n                st.last_hash = None\n                st.cooldown = livelock_cooldown()\n                _STATE["perturbations_injected"] += 1\n            if blocks:\n                return text + "\\n" + "\\n".join(blocks)\n        except Exception:  # noqa: BLE001\n            pass\n        return text\n\n    # -- (c) retry: wrap _chat_completion ----------------------------------\n    def chat_completion(self, messages, **kwargs):\n        attempts = 0\n        budget = None\n        started = None\n        try:\n            attempts = retry_attempts()\n            # F4: the turn\'s request_timeout_seconds is the solver\'s wall\n            # bound (min of analyzer timeout / wall remaining / soft\n            # remaining) — the retry must stay inside it.\n            budget = kwargs.get("request_timeout_seconds")\n            if budget is None:\n                budget = getattr(self, "_timeout", None)\n            budget = None if budget is None else float(budget)\n            started = time.monotonic()\n        except Exception:  # noqa: BLE001\n            attempts = 0\n        if attempts <= 0:\n            return _STOCK["chat_completion"](self, messages, **kwargs)\n        for attempt in range(attempts + 1):\n            try:\n                return _STOCK["chat_completion"](self, messages, **kwargs)\n            except (requests.Timeout, requests.ConnectionError):\n                if attempt >= attempts:\n                    raise\n                try:\n                    backoff = retry_backoff()\n                    if budget is not None:\n                        leftover = budget - (time.monotonic() - started) - backoff\n                        if leftover < retry_min_budget():\n                            # F4: no meaningful wall left — propagate (stock\n                            # behaviour; the solver\'s should_stop-guarded\n                            # retry path takes over).\n                            raise\n                        kwargs = dict(kwargs)\n                        kwargs["request_timeout_seconds"] = leftover\n                    _STATE["retries_used"] += 1\n                    if backoff > 0:\n                        time.sleep(backoff)\n                except (requests.Timeout, requests.ConnectionError):\n                    raise\n                except Exception:  # noqa: BLE001\n                    pass\n        raise RuntimeError("unreachable: TP9 retry loop exhausted without raising")\n\n    analyze._tp9_stock = _STOCK["analyze"]\n    build_user_prompt._tp9_stock = _STOCK["build_user_prompt"]\n    chat_completion._tp9_stock = _STOCK["chat_completion"]\n    cls.analyze = analyze\n    cls._build_user_prompt = build_user_prompt\n    cls._chat_completion = chat_completion\n    _STATE["installed"] = True\n    return "pipeline: OK"\n', 'graft_memoryspine.py': '"""Memory-spine graft (TP10, 2026-08-31) — persistent notes, note-to-self echo,\nhonest last-turn accounting. Rank-2 in the path-to-7 plan.\n\nDesign source: docs/research-2026-08-31/R11-schema-traces-mining.md §3 (the\n95-99% Schema harness re-injects the agent\'s own notes.md every turn, echoes\nits prior intent/note-to-self verbatim, and reports honest last-turn\naccounting). Why it should pay here: R9 §2 forensics — cross-attempt amnesia\n(sp80: 15 attempts, fresh exploration each time) and the 08-29 review\'s 43%\nzero-level plays that are memory/control-bound, not action-starved.\n\nSTOCK BEHAVIOUR (verified in the bundled tool_agent.py) and the TRUE DELTA:\n\n(a) NOTES. Stock already harvests labelled world-model text from the\n    assistant channel (_extract_scientist_note :412, applied :1335) and\n    re-injects the LIVE values into every user prompt\n    (_summarized_knowledge_lines :1358, used :1472) — reinjection per turn is\n    NOT the gap. The gaps are: each harvest OVERWRITES the field (no history);\n    _update_summarized_knowledge_from_step_summary :1343 WIPES every field\n    except cross_level_notes on level_transition/run_complete/game_over\n    (graft_throughput\'s TP_KEEP_NOTES_ON_GAME_OVER suppresses only the\n    game_over wipe); and _ensure_session :1140 clears everything on a session\n    change, so nothing survives into a new pass of the same game.\n    DELTA: a module-level per-GAME journal, synced by diffing\n    _summarized_knowledge (so it also catches graft_emission\'s direct\n    reasoning-channel writes), snapshotted BEFORE the wipe and BEFORE a\n    session reset, and re-injected as a capped tail (TP10_NOTES_CAP, default\n    4000 chars) under "YOUR NOTES (persistent):". Entries identical to the\n    live block are skipped, so on healthy turns the block only carries what\n    stock has lost (earlier levels, pre-wipe state, earlier passes).\n\n(b) SUGGESTION ECHO. Stock maps "Plan:"/"Next test:" into current_plan —\n    also overwritten and wiped; there is no verbatim echo. DELTA: harvest the\n    last one-line `Next:` / `Suggestion:` from assistant text and echo it\n    verbatim at the TOP of the next user prompt ("YOUR PRIOR INTENT: ...");\n    consumed after one echo so a stale note is never re-served as fresh.\n    J10-F1: the harvested line is STRIPPED from the text handed to the stock\n    harvest — _extract_labeled_blocks (:375-:408) glues any unlabeled line\n    into the preceding labeled block, so an unstripped `Next:` line would be\n    absorbed into current_plan/world_model and re-served every turn as a\n    standing plan (and journaled as a stale imperative).\n\n(c) HONEST ACCOUNTING. Stock\'s prompt header (:1416) reports executed count\n    and names but never committed-vs-executed: requested_count /\n    stopped_early / state ARE recorded per action() payload (:329-:338,\n    :1674-:1682) and then dropped by _summarize_step_sequence (:1250) —\n    stop_reason and level ARE kept by the stock summary (:1288, :1293).\n    DELTA: augment the step summary with the committed total and end state\n    from the SAME recorded payloads, and prepend one line:\n    "LAST TURN: committed N action(s), M executed, ended level=L, state=S."\n    plus how many were dropped and the recorded stop_reason when a batch was\n    cut short. No mispredict detection is invented — only recorded fields.\n    J10-F2: requested_count is computed AFTER graft_throughput\'s batch-cap\n    truncation (:1760, :1890 sit downstream of _normalize_python_actions,\n    which TP wraps to truncate), so requested_count alone under-reports what\n    the model committed. TP10 therefore also wraps _normalize_python_actions\n    — whose stock either normalizes EVERY item or raises, never partially\n    drops — to record the RAW batch size per action() call (works in both\n    install orders because TP\'s cap wrapper hands the raw value to the inner\n    chain before truncating), and reports cap-truncated actions explicitly\n    ("N truncated by the harness batch cap").\n\nSeams (all rebinding; stock callables in the module-level _STOCK dict so\nstacked wrappers survive later installs):\n  ToolAgent._build_user_prompt                       — inject (a)(b)(c)\n  ToolAgent._update_summarized_knowledge_from_assistant — harvest (a)(b),\n                                                       intent line stripped first\n  ToolAgent._update_summarized_knowledge_from_step_summary — pre-wipe snapshot\n  ToolAgent._ensure_session                          — game key + pre-reset snapshot\n  ToolAgent._summarize_step_sequence                 — committed/state fields\n  ToolAgent._normalize_python_actions                — raw pre-clamp batch size\n  ToolAgent._run_python_tool                         — per-tool-call raw-count reset\n\nFlags (read at call time): TP10_ENABLE=1 master ("0" = pass-through);\nTP10_NOTES=1, TP10_ECHO=1, TP10_ACCOUNTING=1 per behaviour;\nTP10_NOTES_CAP=2000 chars of injected notes tail (J10-F4: R11 §3 prescribes\n<=2KB for the A/B; 4000 measured -4 retained history turns under 32k);\nTP10_HOWTO_EVERY=8 — the note-to-self how-to line rides prompt 1, every Nth\nprompt after, and the first prompt after a knowledge wipe (0 = wipe-only).\nFail-open: every graft path is try/except\'d back to stock output.\n"""\nfrom __future__ import annotations\n\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any\n\n_STATE = {"installed": False}\n_STOCK: dict[str, Any] = {}\n_OFF = {"0", "false", "no", "off"}\n\n# game_key -> chronological journal of {"key","label","level","text"}\n_NOTES: dict[str, list[dict[str, Any]]] = {}\n_JOURNAL_MAX = 500\n_JOURNAL_TRIM_TO = 400\n_INTENT_MAX = 300\n\n_LABELS = (\n    ("world_model", "World model"),\n    ("goal_model", "Goal model"),\n    ("action_model", "Action model"),\n    ("recent_findings", "Recent findings"),\n    ("open_questions", "Open questions"),\n    ("current_plan", "Plan"),\n    ("cross_level_notes", "Cross-level notes"),\n)\n\nNOTES_HEADER = (\n    "YOUR NOTES (persistent): saved from your earlier world-model updates in THIS game; "\n    "they survive GAME_OVER and level changes. Reuse them instead of re-discovering."\n)\nECHO_LINE = (\n    "YOUR PRIOR INTENT (your note-to-self from last turn — reconsider, don\'t just obey): {intent}"\n)\nECHO_HOWTO = (\n    "To leave a note-to-self for your next turn, put one line starting `Next:` "\n    "(or `Suggestion:`) in your assistant text; it will be echoed back to you verbatim."\n)\n\n\n# ----------------------------------------------------------------- flags ---\ndef _env(name: str, default: str) -> str:\n    raw = os.environ.get(name)\n    return default if raw is None or not raw.strip() else raw.strip()\n\n\ndef enabled() -> bool:\n    return _env("TP10_ENABLE", "1").lower() not in _OFF\n\n\ndef notes_enabled() -> bool:\n    return enabled() and _env("TP10_NOTES", "1").lower() not in _OFF\n\n\ndef echo_enabled() -> bool:\n    return enabled() and _env("TP10_ECHO", "1").lower() not in _OFF\n\n\ndef accounting_enabled() -> bool:\n    return enabled() and _env("TP10_ACCOUNTING", "1").lower() not in _OFF\n\n\ndef notes_cap() -> int:\n    try:\n        return max(200, int(_env("TP10_NOTES_CAP", "2000")))\n    except ValueError:\n        return 2000\n\n\ndef howto_every() -> int:\n    try:\n        return max(0, int(_env("TP10_HOWTO_EVERY", "8")))\n    except ValueError:\n        return 8\n\n\ndef status() -> dict[str, Any]:\n    return {\n        "installed": _STATE["installed"],\n        "enabled": enabled(),\n        "notes": notes_enabled(),\n        "echo": echo_enabled(),\n        "accounting": accounting_enabled(),\n        "notes_cap": notes_cap(),\n        "howto_every": howto_every(),\n        "games_tracked": len(_NOTES),\n    }\n\n\n# ------------------------------------------------------------- game key ---\ndef _game_key(state_path: Any) -> str:\n    p = Path(state_path)\n    stem = p.stem  # e.g. "<artifact_stem>_p0_tool_runtime_state"\n    base = re.sub(r"_p\\d+_.*$", "", stem)\n    if base == stem:\n        base = re.sub(r"_p\\d+$", "", stem)\n    return f"{p.parent}::{base or stem}"\n\n\ndef game_key_of(agent: Any) -> str:\n    key = getattr(agent, "_tp10_game_key", None)\n    return key if key else f"agent-{id(agent)}"\n\n\ndef _summary_level(agent: Any) -> int | None:\n    try:\n        summary = getattr(agent, "_last_step_summary", None) or {}\n        level = summary.get("level")\n        return int(level) if level is not None else None\n    except Exception:  # noqa: BLE001\n        return None\n\n\n# --------------------------------------------------------- notes journal ---\ndef _latest_for_key(journal: list[dict[str, Any]], key: str) -> dict[str, Any] | None:\n    for entry in reversed(journal):\n        if entry.get("key") == key:\n            return entry\n    return None\n\n\ndef sync_journal(agent: Any, level: int | None) -> None:\n    """Diff the live _summarized_knowledge into the per-game journal.\n\n    Diff-based (not harvest-hook-based) so notes written by ANY channel —\n    stock assistant harvest, graft_emission\'s reasoning-channel fallback,\n    direct writes — are captured before a wipe can destroy them.\n    """\n    if not notes_enabled():\n        return\n    try:\n        knowledge = getattr(agent, "_summarized_knowledge", None)\n        if not isinstance(knowledge, dict):\n            return\n        journal = _NOTES.setdefault(game_key_of(agent), [])\n        for key, label in _LABELS:\n            text = str(knowledge.get(key) or "").strip()\n            if not text:\n                continue\n            latest = _latest_for_key(journal, key)\n            if latest is not None and latest.get("text") == text:\n                continue\n            journal.append({"key": key, "label": label, "level": level, "text": text})\n        if len(journal) > _JOURNAL_MAX:\n            del journal[: len(journal) - _JOURNAL_TRIM_TO]\n    except Exception:  # noqa: BLE001\n        pass\n\n\ndef render_notes(agent: Any) -> str | None:\n    journal = _NOTES.get(game_key_of(agent))\n    if not journal:\n        return None\n    live = getattr(agent, "_summarized_knowledge", None) or {}\n    seen: set[tuple[Any, Any]] = set()\n    keys_seen: set[Any] = set()\n    selected: list[dict[str, Any]] = []\n    for entry in reversed(journal):  # newest -> oldest\n        key = entry.get("key")\n        level = entry.get("level")\n        # J10-F3: a pre-first-summary (level=None) note is superseded by ANY\n        # newer entry for the same key — don\'t re-serve refuted early guesses\n        # beside their own corrections.\n        if level is None and key in keys_seen:\n            continue\n        ident = (key, level)\n        if ident in seen:\n            continue\n        seen.add(ident)\n        keys_seen.add(key)\n        # already shown verbatim in the live world-model block -> skip\n        if str(live.get(entry.get("key")) or "").strip() == entry.get("text"):\n            continue\n        selected.append(entry)\n    if not selected:\n        return None\n    selected.reverse()  # chronological, newest last\n    lines = []\n    for entry in selected:\n        level = entry.get("level")\n        tag = f"[L{level}] " if level is not None else ""\n        lines.append(f"- {tag}{entry.get(\'label\')}: {entry.get(\'text\')}")\n    cap = notes_cap()\n    kept: list[str] = []\n    total = 0\n    trimmed = False\n    for line in reversed(lines):  # keep the newest tail within the cap\n        if kept and total + len(line) + 1 > cap:\n            trimmed = True\n            break\n        if not kept and len(line) + 1 > cap:\n            line = line[: max(0, cap - 16)].rstrip() + "... [truncated]"\n        kept.append(line)\n        total += len(line) + 1\n    kept.reverse()\n    header = NOTES_HEADER + (" (older notes trimmed)" if trimmed else "")\n    return header + "\\n" + "\\n".join(kept)\n\n\n# ------------------------------------------------------- suggestion echo ---\ndef split_intent(content: str) -> tuple[str, str | None]:\n    """(text with `Next:`/`Suggestion:` lines removed, last one-line note).\n\n    J10-F1: the intent line MUST be removed from the text the stock harvest\n    sees — _extract_labeled_blocks glues unlabeled lines into the preceding\n    labeled block, so an unstripped note-to-self would be absorbed into\n    current_plan/world_model, re-served every turn, and journaled.\n    """\n    if not content or not content.strip():\n        return content, None\n    found: str | None = None\n    kept: list[str] = []\n    for raw_line in content.splitlines():\n        line = raw_line.strip()\n        while line[:1] in {"-", "*"}:\n            line = line[1:].lstrip()\n        lowered = line.lower()\n        matched = False\n        for prefix in ("next:", "suggestion:"):\n            if lowered.startswith(prefix):\n                matched = True\n                value = line[len(prefix):].strip()\n                if value:\n                    found = value\n                break\n        if not matched:\n            kept.append(raw_line)\n    if found and len(found) > _INTENT_MAX:\n        found = found[:_INTENT_MAX].rstrip() + "..."\n    return ("\\n".join(kept), found) if found is not None else (content, None)\n\n\ndef harvest_intent(content: str) -> str | None:\n    """Last one-line `Next:` / `Suggestion:` note in the assistant text."""\n    return split_intent(content)[1]\n\n\n# ----------------------------------------------------- honest accounting ---\ndef accounting_line(summary: Any) -> str | None:\n    """One line from the recorded per-turn payload fields only."""\n    if not isinstance(summary, dict) or not summary:\n        return None\n    try:\n        executed = int(summary.get("executed_count") or 0)\n    except (TypeError, ValueError):\n        return None\n    try:\n        committed = int(summary.get("tp10_committed"))\n    except (TypeError, ValueError):\n        committed = executed\n    committed = max(committed, executed)\n    level = summary.get("level")\n    state = summary.get("tp10_state")\n    if not state:\n        if summary.get("game_over"):\n            state = "GAME_OVER"\n        elif summary.get("run_complete"):\n            state = "WIN"\n        else:\n            state = "NOT_FINISHED"\n    line = (\n        f"LAST TURN: committed {committed} action(s), {executed} executed, "\n        f"ended level={level}, state={state}."\n    )\n    dropped = committed - executed\n    if dropped > 0:\n        try:\n            cap_dropped = min(dropped, max(0, int(summary.get("tp10_cap_dropped") or 0)))\n        except (TypeError, ValueError):\n            cap_dropped = 0\n        details = []\n        if cap_dropped > 0:\n            details.append(f"{cap_dropped} truncated by the harness batch cap")\n        reason = str(summary.get("stop_reason") or "").strip()\n        if reason and dropped - cap_dropped > 0:\n            details.append(f"stop_reason={reason}")\n        line += f" {dropped} committed action(s) were dropped before execution"\n        line += f" ({\'; \'.join(details)})." if details else "."\n    return line\n\n\n# --------------------------------------------------------------- install ---\ndef install() -> str:\n    if _STATE["installed"]:\n        return "memoryspine: SKIP (already applied)"\n    try:\n        from inference.agent import tool_agent as agent_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"memoryspine: SKIP (import failed: {exc!r})"\n    cls = getattr(agent_mod, "ToolAgent", None)\n    if cls is None:\n        return "memoryspine: SKIP (ToolAgent missing)"\n    for name in (\n        "_build_user_prompt",\n        "_update_summarized_knowledge_from_assistant",\n        "_update_summarized_knowledge_from_step_summary",\n        "_ensure_session",\n        "_summarize_step_sequence",\n        "_normalize_python_actions",\n        "_run_python_tool",\n    ):\n        if getattr(cls, name, None) is None:\n            return f"memoryspine: SKIP ({name} seam missing)"\n\n    _STOCK["build_user_prompt"] = cls._build_user_prompt\n    _STOCK["update_from_assistant"] = cls._update_summarized_knowledge_from_assistant\n    _STOCK["update_from_step_summary"] = cls._update_summarized_knowledge_from_step_summary\n    _STOCK["ensure_session"] = cls._ensure_session\n    _STOCK["summarize_step_sequence"] = cls._summarize_step_sequence\n    _STOCK["normalize_python_actions"] = cls._normalize_python_actions\n    _STOCK["run_python_tool"] = cls._run_python_tool\n\n    # -- seam: session identity + pre-reset snapshot ------------------------\n    def ensure_session(self, state_path):\n        try:\n            if enabled():\n                old_dir = getattr(self, "_session_runtime_dir", None)\n                new_dir = getattr(Path(state_path), "parent", None)\n                if old_dir is not None and new_dir is not None and old_dir != new_dir:\n                    # stock is about to clear _summarized_knowledge: snapshot\n                    # it under the OLD game key first.\n                    sync_journal(self, _summary_level(self))\n        except Exception:  # noqa: BLE001\n            pass\n        result = _STOCK["ensure_session"](self, state_path)\n        try:\n            self._tp10_game_key = _game_key(state_path)\n        except Exception:  # noqa: BLE001\n            pass\n        return result\n\n    ensure_session._tp10_stock = _STOCK["ensure_session"]\n    cls._ensure_session = ensure_session\n\n    # -- seam: harvest (notes sync + intent) --------------------------------\n    def update_from_assistant(self, content):\n        text = content\n        intent = None\n        if enabled() and echo_enabled():\n            try:\n                # J10-F1: remove the note-to-self line BEFORE the stock\n                # harvest sees it, or _extract_labeled_blocks glues it into\n                # the preceding labeled block for the rest of the game.\n                stripped, intent = split_intent(str(content or ""))\n                if intent is not None:\n                    text = stripped\n            except Exception:  # noqa: BLE001\n                text, intent = content, None\n        result = _STOCK["update_from_assistant"](self, text)\n        if not enabled():\n            return result\n        try:\n            if intent:\n                self._tp10_intent = intent\n            sync_journal(self, _summary_level(self))\n        except Exception:  # noqa: BLE001\n            pass\n        return result\n\n    update_from_assistant._tp10_stock = _STOCK["update_from_assistant"]\n    cls._update_summarized_knowledge_from_assistant = update_from_assistant\n\n    # -- seam: pre-wipe snapshot --------------------------------------------\n    def update_from_step_summary(self):\n        if enabled():\n            try:\n                summary = getattr(self, "_last_step_summary", None) or {}\n                level = _summary_level(self)\n                if summary.get("level_transition") and isinstance(level, int) and level > 1:\n                    # the knowledge being wiped described the level just left\n                    level = level - 1\n                sync_journal(self, level)\n                if summary.get("level_transition") or summary.get("run_complete") or summary.get("game_over"):\n                    # J10-F4: re-teach the note-to-self affordance on the\n                    # first prompt after a knowledge wipe\n                    self._tp10_wipe_pending = True\n            except Exception:  # noqa: BLE001\n                pass\n        return _STOCK["update_from_step_summary"](self)\n\n    update_from_step_summary._tp10_stock = _STOCK["update_from_step_summary"]\n    cls._update_summarized_knowledge_from_step_summary = update_from_step_summary\n\n    # -- seam: raw pre-clamp batch size (J10-F2) ----------------------------\n    # graft_throughput\'s batch cap truncates inside its _normalize_python_actions\n    # wrapper, and requested_count (:1760, :1890) is computed AFTER that — so\n    # requested_count alone under-reports what the model committed. Stock\n    # normalize either accepts every item or raises (:1614-:1650, no partial\n    # drop), and TP\'s cap wrapper hands the raw value to the inner chain\n    # before truncating, so this wrapper sees the true batch size in BOTH\n    # install orders. Recorded only when the chain returns (a cap REFUSAL\n    # raises out of the outer wrapper and the model gets the explicit error).\n    def normalize_python_actions(self, value):\n        result = _STOCK["normalize_python_actions"](self, value)\n        if enabled():\n            try:\n                if isinstance(value, (list, tuple)):\n                    raw = len(value)\n                elif isinstance(value, (str, dict)):\n                    raw = 1\n                else:\n                    raw = len(result)\n                calls = getattr(self, "_tp10_raw_committed", None)\n                if not isinstance(calls, list):\n                    # lazily create so the capture also works when a caller\n                    # bypasses _run_python_tool (the run wrapper still resets\n                    # the list at the top of every real tool call)\n                    calls = []\n                    self._tp10_raw_committed = calls\n                calls.append(max(int(raw), len(result)))\n            except Exception:  # noqa: BLE001\n                pass\n        return result\n\n    normalize_python_actions._tp10_stock = _STOCK["normalize_python_actions"]\n    cls._normalize_python_actions = normalize_python_actions\n\n    def run_python_tool(self, state_path, arguments):\n        try:\n            self._tp10_raw_committed = []\n        except Exception:  # noqa: BLE001\n            pass\n        return _STOCK["run_python_tool"](self, state_path, arguments)\n\n    run_python_tool._tp10_stock = _STOCK["run_python_tool"]\n    cls._run_python_tool = run_python_tool\n\n    # -- seam: committed/state ride the recorded step summary ---------------\n    def summarize_step_sequence(self, action_results):\n        summary = _STOCK["summarize_step_sequence"](self, action_results)\n        if not enabled() or summary is None:\n            return summary\n        try:\n            committed = 0\n            state = None\n            for item in action_results or []:\n                if not isinstance(item, dict):\n                    continue\n                requested = item.get("requested_count")\n                if requested is None:\n                    requested = item.get("executed_count")\n                if requested is None:\n                    requested = 1\n                try:\n                    committed += max(0, int(requested))\n                except (TypeError, ValueError):\n                    committed += 1\n                if item.get("executed") and item.get("state") is not None:\n                    state = item.get("state")\n            # J10-F2: prefer the true pre-clamp count captured at the\n            # normalize seam; the difference is what the batch cap discarded.\n            raw_calls = getattr(self, "_tp10_raw_committed", None)\n            if isinstance(raw_calls, list) and raw_calls:\n                raw_total = sum(int(n) for n in raw_calls)\n                if raw_total > committed:\n                    summary["tp10_cap_dropped"] = raw_total - committed\n                    committed = raw_total\n            summary["tp10_committed"] = committed\n            if state is not None:\n                summary["tp10_state"] = str(state)\n        except Exception:  # noqa: BLE001\n            pass\n        return summary\n\n    summarize_step_sequence._tp10_stock = _STOCK["summarize_step_sequence"]\n    cls._summarize_step_sequence = summarize_step_sequence\n\n    # -- seam: prompt injection ---------------------------------------------\n    def build_user_prompt(self, action_num, *args, **kwargs):\n        text = _STOCK["build_user_prompt"](self, action_num, *args, **kwargs)\n        if not enabled():\n            return text\n        try:\n            prefix: list[str] = []\n            if accounting_enabled():\n                line = accounting_line(getattr(self, "_last_step_summary", None))\n                if line:\n                    prefix.append(line)\n            if echo_enabled():\n                intent = getattr(self, "_tp10_intent", None)\n                if intent:\n                    prefix.append(ECHO_LINE.format(intent=intent))\n                    self._tp10_intent = None  # one echo per note; never re-serve stale intent\n            suffix: list[str] = []\n            if notes_enabled():\n                # final sync catches direct writes (e.g. TP5 reasoning harvest)\n                sync_journal(self, _summary_level(self))\n                block = render_notes(self)\n                if block:\n                    suffix.append(block)\n            if echo_enabled():\n                # J10-F4: the how-to line is ~150 chars on EVERY turn if\n                # unconditional; ride prompt 1, every Nth prompt, and the\n                # first prompt after a knowledge wipe.\n                count = int(getattr(self, "_tp10_prompt_count", 0) or 0) + 1\n                self._tp10_prompt_count = count\n                every = howto_every()\n                periodic = every > 0 and (count - 1) % every == 0\n                if periodic or bool(getattr(self, "_tp10_wipe_pending", False)):\n                    suffix.append(ECHO_HOWTO)\n                    self._tp10_wipe_pending = False\n            if not prefix and not suffix:\n                return text\n            return "\\n".join([*prefix, text, *suffix])\n        except Exception:  # noqa: BLE001\n            return text\n\n    build_user_prompt._tp10_stock = _STOCK["build_user_prompt"]\n    cls._build_user_prompt = build_user_prompt\n\n    _STATE["installed"] = True\n    return "memoryspine: OK"\n'}
for _name, _src in _GRAFT_SOURCES.items():
    (_G_DIR / _name).write_text(_src, encoding="utf-8")
if str(_G_DIR) not in sys.path:
    sys.path.insert(0, str(_G_DIR))
_installed = {}
for _mod_name in ['graft_throughput', 'graft_control', 'graft_explore', 'graft_emission', 'graft_economy', 'graft_deaths', 'graft_durable', 'graft_pipeline', 'graft_memoryspine']:
    _m = _il.import_module(_mod_name)
    _st = _m.install()
    _installed[_mod_name] = _st
    assert _st.endswith(": OK"), _st
print("[ab] grafts installed:", _installed, flush=True)


In [ ]:
# ==== two-phase A/B run: stock then tp10 (one boot, same server) ====
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))
assert not TRUE_SUBMISSION, "A/B smoke kernel must never run as a submission"

_ENV_DIR = str(Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent / "environment_files")
PHASES = [
    ("stock", {f: "0" for f in ['TP_ENABLE', 'TP2_ENABLE', 'TP4_ENABLE', 'TP5_ENABLE', 'TP6_ENABLE', 'TP7_ENABLE', 'TP8_ENABLE', 'TP9_ENABLE', 'TP10_ENABLE']}),
    ("tp10", dict({f: "0" for f in ['TP_ENABLE', 'TP2_ENABLE', 'TP4_ENABLE', 'TP5_ENABLE', 'TP6_ENABLE', 'TP7_ENABLE', 'TP8_ENABLE', 'TP9_ENABLE', 'TP10_ENABLE']}, **{'TP10_ENABLE': '1'})),
]
AB_RESULTS = {}
_PHASE_BUDGET_S = 4.0 * 3600

_v31_start_watchdog(bm.solver)
try:
    for _phase_name, _phase_env in PHASES:
        for _k, _v in _phase_env.items():
            os.environ[_k] = _v
        bm.game_runs = []
        bm.games = _offline_games(_ENV_DIR)
        bm.n_passes = 1
        bm.game_weights = None
        bm.label = "v22-ab-" + _phase_name
        _soft_end = datetime.now() + timedelta(seconds=_PHASE_BUDGET_S)
        print(f"=== PHASE {_phase_name}: {len(bm.games)} games env={_phase_env} "
              f"soft_end={_soft_end} ===", flush=True)
        try:
            await bm.run(soft_end_time=_soft_end, runtime_environment=target,
                         minimal_diagnostics=False)
        except Exception as _exc:  # noqa: BLE001
            import traceback

            print(f"PHASE {_phase_name} RAISED {type(_exc).__name__}: {_exc}", flush=True)
            traceback.print_exc()
        games = []
        for game_run in list(bm.game_runs):
            apl = list(game_run.actions_per_level or [])
            games.append({
                "game_id": game_run.game_id,
                "state": str(game_run.state),
                "levels_completed": game_run.levels_completed,
                "number_of_levels": game_run.number_of_levels,
                "final_score": game_run.final_score,
                "actions": sum(apl) if apl else len(game_run.history),
                "actions_per_level": apl,
                "wallclock_s": game_run.final_wallclock_seconds,
            })
        n = max(1, len(games))
        AB_RESULTS[_phase_name] = {
            "games": games,
            "n": len(games),
            "mean_score": round(sum(g["final_score"] for g in games) / n, 3),
            "mean_levels": round(sum(g["levels_completed"] for g in games) / n, 3),
            "zero_level": sum(1 for g in games if not g["levels_completed"]),
            "total_actions": sum(g["actions"] for g in games),
        }
        (WORKING_DIR / "ab_results.json").write_text(json.dumps(AB_RESULTS, indent=1))
        print(f"=== PHASE READ {_phase_name}: mean_score={AB_RESULTS[_phase_name]['mean_score']} "
              f"mean_levels={AB_RESULTS[_phase_name]['mean_levels']} "
              f"zero={AB_RESULTS[_phase_name]['zero_level']}/{n} "
              f"actions={AB_RESULTS[_phase_name]['total_actions']} ===", flush=True)
finally:
    _v31_stop_watchdog()
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print(f"taaf.kaggle: teardown command: {command}", flush=True)
        subprocess.run(command, shell=True, check=False, cwd=WORKING_DIR, env=_command_env())

print("\n==== A/B SUMMARY (tp10 vs stock) ====")
for _p, _r in AB_RESULTS.items():
    print(f"{_p:8s} mean_score={_r['mean_score']} mean_levels={_r['mean_levels']} "
          f"zero={_r['zero_level']}/{_r['n']} actions={_r['total_actions']}")
